In [1]:
!pip install -q pandas requests beautifulsoup4 lxml openpyxl ddgs

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import re
import time
from urllib.parse import urljoin, urlparse

In [3]:
columns = [
    "State",
    "Region",
    "Country",
    "Name",
    "Website",
    "Email",
    "Contact Number",
    "Coverage State",
    "Coverage Type",
    "Services Rendered",
    "Target Customers",
    "Source",
    "Source URL"
]

health_df = pd.DataFrame(columns=columns)

health_df

,State,Region,Country,Name,Website,Email,Contact Number,Coverage State,Coverage Type,Services Rendered,Target Customers,Source,Source URL


In [4]:
companies = [
    ("England", "Bupa", "https://www.bupa.co.uk/"),
    ("England", "AXA Health", "https://www.axahealth.co.uk/"),
    ("England", "Aviva", "https://www.aviva.co.uk/"),
    ("England", "WPA", "https://www.wpa.org.uk/"),
    ("England", "VitalityHealth", "https://www.vitality.co.uk/"),
    ("England", "The Exeter", "https://www.the-exeter.com/"),
    ("England", "Benenden Health", "https://www.benenden.co.uk/"),
    ("England", "Freedom Health Insurance", "https://www.freedomhealthinsurance.co.uk/"),
    ("England", "National Friendly", "https://www.nationalfriendly.co.uk/"),
    ("England", "Saga Health Insurance", "https://www.saga.co.uk/health-insurance"),
    ("England", "Cigna Healthcare", "https://www.cigna.co.uk/"),
    ("England", "Simplyhealth", "https://www.simplyhealth.co.uk/"),
    ("England", "General & Medical", "https://www.gminsure.com/"),
    ("England", "CS Healthcare", "https://www.cshealthcare.co.uk/")
]

In [5]:
health_df = pd.DataFrame([
    {
        "State": state,
        "Region": "N/A",
        "Country": "United Kingdom",
        "Name": name,
        "Website": website,
        "Email": "N/A",
        "Contact Number": "N/A",
        "Coverage State": "N/A",
        "Coverage Type": "N/A",
        "Services Rendered": "N/A",
        "Target Customers": "N/A",
        "Source": "Official Company Website",
        "Source URL": website
    }
    for state, name, website in companies
])

health_df

,State,Region,Country,Name,Website,Email,Contact Number,Coverage State,Coverage Type,Services Rendered,Target Customers,Source,Source URL
0,England,N/A,United Kingdom,Bupa,https://www.bupa.co.uk/,N/A,N/A,N/A,N/A,N/A,N/A,Official Company Website,https://www.bupa.co.uk/
1,England,N/A,United Kingdom,AXA Health,https://www.axahealth.co.uk/,N/A,N/A,N/A,N/A,N/A,N/A,Official Company Website,https://www.axahealth.co.uk/
2,England,N/A,United Kingdom,Aviva,https://www.aviva.co.uk/,N/A,N/A,N/A,N/A,N/A,N/A,Official Company Website,https://www.aviva.co.uk/
3,England,N/A,United Kingdom,WPA,https://www.wpa.org.uk/,N/A,N/A,N/A,N/A,N/A,N/A,Official Company Website,https://www.wpa.org.uk/
4,England,N/A,United Kingdom,VitalityHealth,https://www.vitality.co.uk/,N/A,N/A,N/A,N/A,N/A,N/A,Official Company Website,https://www.vitality.co.uk/
5,England,N/A,United Kingdom,The Exeter,https://www.the-exeter.com/,N/A,N/A,N/A,N/A,N/A,N/A,Official Company Website,https://www.the-exeter.com/
6,England,N/A,United Kingdom,Benenden Health,https://www.benenden.co.uk/,N/A,N/A,N/A,N/A,N/A,N/A,Official Company Website,https://www.benenden.co.uk/
7,England,N/A,United Kingdom,Freedom Health Insurance,https://www.freedomhealthinsurance.co.uk/,N/A,N/A,N/A,N/A,N/A,N/A,Official Company Website,https://www.freedomhealthinsurance.co.uk/
8,England,N/A,United Kingdom,National Friendly,https://www.nationalfriendly.co.uk/,N/A,N/A,N/A,N/A,N/A,N/A,Official Company Website,https://www.nationalfriendly.co.uk/
9,England,N/A,United Kingdom,Saga Health Insurance,https://www.saga.co.uk/health-insurance,N/A,N/A,N/A,N/A,N/A,N/A,Official Company Website,https://www.saga.co.uk/health-insurance


In [6]:
def scrape_website(url):
    try:
        response = requests.get(
            url,
            headers={
                "User-Agent": "Mozilla/5.0"
            },
            timeout=15
        )

        if response.status_code != 200:
            return None

        return BeautifulSoup(response.text, "lxml")

    except:
        return None

In [7]:
def find_phones(soup):
    if soup is None:
        return []

    text = soup.get_text(" ", strip=True)

    phones = re.findall(
        r'(?:\+44\s?\(?0?\)?\s?|0)(?:\d[\s\-]?){9,12}',
        text
    )

    cleaned = []

    for phone in phones:
        phone = re.sub(r'\s+', ' ', phone).strip()

        if phone not in cleaned:
            cleaned.append(phone)

    return cleaned

In [9]:
def find_emails(soup):
    if soup is None:
        return []

    text = soup.get_text(" ", strip=True)

    emails = re.findall(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        text
    )

    return list(dict.fromkeys(emails))

In [10]:
for i, row in health_df.iterrows():

    print("Processing:", row["Name"])

    soup = scrape_website(row["Website"])

    emails = find_emails(soup)
    phones = find_phones(soup)

    if emails:
        health_df.at[i, "Email"] = emails[0]

    if phones:
        health_df.at[i, "Contact Number"] = phones[0]

    time.sleep(1)

Processing: Bupa
Processing: AXA Health
Processing: Aviva
Processing: WPA
Processing: VitalityHealth
Processing: The Exeter
Processing: Benenden Health
Processing: Freedom Health Insurance
Processing: National Friendly
Processing: Saga Health Insurance
Processing: Cigna Healthcare
Processing: Simplyhealth
Processing: General & Medical
Processing: CS Healthcare


In [11]:
health_df[
    [
        "Name",
        "Website",
        "Email",
        "Contact Number"
    ]
]

,Name,Website,Email,Contact Number
0,Bupa,https://www.bupa.co.uk/,N/A,N/A
1,AXA Health,https://www.axahealth.co.uk/,N/A,0800 169 7593
2,Aviva,https://www.aviva.co.uk/,N/A,N/A
3,WPA,https://www.wpa.org.uk/,N/A,N/A
4,VitalityHealth,https://www.vitality.co.uk/,N/A,N/A
5,The Exeter,https://www.the-exeter.com/,enquiries@the-exeter.com,0300 123 3201
6,Benenden Health,https://www.benenden.co.uk/,N/A,N/A
7,Freedom Health Insurance,https://www.freedomhealthinsurance.co.uk/,N/A,01202 756 350
8,National Friendly,https://www.nationalfriendly.co.uk/,N/A,N/A
9,Saga Health Insurance,https://www.saga.co.uk/health-insurance,N/A,0808 239 3479


In [12]:
england_df = health_df.copy()
print("England records:", len(england_df))

England records: 14


In [13]:
from ddgs import DDGS
import time

def search_company(company, query):
    try:
        results = list(
            DDGS().text(
                f'"{company}" {query} official UK',
                max_results=5
            )
        )
        return results
    except Exception as e:
        print(f"Search failed for {company}: {e}")
        return []

contact_results = {}

for company in england_df["Name"]:
    print("Searching:", company)

    results = search_company(company, "contact phone email")
    contact_results[company] = results

    time.sleep(1)

print("Done.")

Searching: Bupa
Searching: AXA Health
Searching: Aviva
Searching: WPA
Searching: VitalityHealth
Searching: The Exeter
Searching: Benenden Health
Searching: Freedom Health Insurance
Searching: National Friendly
Searching: Saga Health Insurance
Searching: Cigna Healthcare
Searching: Simplyhealth
Searching: General & Medical
Searching: CS Healthcare
Done.


In [14]:
for company, results in contact_results.items():
    print("\n==============================")
    print(company)
    
    for r in results[:3]:
        print("TITLE:", r.get("title"))
        print("URL:", r.get("href"))
        print("DESCRIPTION:", r.get("body"))


Bupa
TITLE: Contact us | Ways to get in touch with Bupa UK
URL: https://www.bupa.co.uk/contact-us.php
DESCRIPTION: Discover contact information for Bupa UK for both existing and new customers. For phone numbers and app information, visit our contact web page.
TITLE: Contact Us | Small Business Healthcare | Bupa UK
URL: https://www.bupa.co.uk/business/small-business-healthcare/small-business-contact-us
DESCRIPTION: Whether you're looking for healthcare for your small business or have existing cover with Bupa, you can talk to us by phone, email or requesting a callback.
TITLE: Bupa UK - Company Profile & Staff Directory | ContactOut
URL: https://contactout.com/company/Bupa-UK-11275
DESCRIPTION: Bupa UK is a Wellness and Fitness Services company located in London, England, United Kingdom with 1,350 employees. Access Bupa UK's email format and staff directory for direct contact details.

AXA Health
TITLE: Contact us | AXA UK | AXA UK home page
URL: https://www.axa.co.uk/contact-us/
DESCRI

In [15]:
contact_updates = {
    "Bupa": {
        "Contact Number": "0345 609 0111",
        "Source URL": "https://www.bupa.co.uk/contact-us.php"
    },

    "AXA Health": {
        "Contact Number": "0800 027 1384",
        "Source URL": "https://www.axahealth.co.uk/health-insurance/contact-us/"
    },

    "Aviva": {
        "Contact Number": "N/A",
        "Source URL": "https://www.aviva.co.uk/help-and-support/contact-us/"
    },

    "WPA": {
        "Contact Number": "N/A",
        "Source URL": "https://www.wpa.org.uk/"
    },

    "VitalityHealth": {
        "Contact Number": "N/A",
        "Source URL": "https://www.vitality.co.uk/"
    },

    "The Exeter": {
        "Email": "member@the-exeter.com",
        "Contact Number": "0300 123 3201",
        "Source URL": "https://the-exeter.com/contact-us/member/"
    },

    "Benenden Health": {
        "Contact Number": "N/A",
        "Source URL": "https://www.benenden.co.uk/contact-us/"
    },

    "Freedom Health Insurance": {
        "Contact Number": "0800 999 2013",
        "Source URL": "https://www.freedomhealthinsurance.co.uk/contact-us"
    },

    "National Friendly": {
        "Contact Number": "N/A",
        "Source URL": "https://www.nationalfriendly.co.uk/"
    },

    "Saga Health Insurance": {
        "Contact Number": "N/A",
        "Source URL": "https://www.saga.co.uk/contact-us/insurance/health-insurance"
    },

    "Cigna Healthcare": {
        "Contact Number": "N/A",
        "Source URL": "https://www.cigna.co.uk/"
    },

    "Simplyhealth": {
        "Contact Number": "0370 908 3304",
        "Source URL": "https://www.simplyhealth.co.uk/contact-us"
    },

    "General & Medical": {
        "Contact Number": "0800 970 9442",
        "Source URL": "https://www.generalandmedical.com/contact-us/"
    },

    "CS Healthcare": {
        "Contact Number": "N/A",
        "Source URL": "https://www.cshealthcare.co.uk/contact-us/"
    }
}

In [16]:
for company, info in contact_updates.items():
    idx = england_df.index[england_df["Name"] == company]

    for column, value in info.items():
        england_df.loc[idx, column] = value

england_df[["Name", "Website", "Email", "Contact Number", "Source URL"]]

,Name,Website,Email,Contact Number,Source URL
0,Bupa,https://www.bupa.co.uk/,N/A,0345 609 0111,https://www.bupa.co.uk/contact-us.php
1,AXA Health,https://www.axahealth.co.uk/,N/A,0800 027 1384,https://www.axahealth.co.uk/health-insurance/c...
2,Aviva,https://www.aviva.co.uk/,N/A,N/A,https://www.aviva.co.uk/help-and-support/conta...
3,WPA,https://www.wpa.org.uk/,N/A,N/A,https://www.wpa.org.uk/
4,VitalityHealth,https://www.vitality.co.uk/,N/A,N/A,https://www.vitality.co.uk/
5,The Exeter,https://www.the-exeter.com/,member@the-exeter.com,0300 123 3201,https://the-exeter.com/contact-us/member/
6,Benenden Health,https://www.benenden.co.uk/,N/A,N/A,https://www.benenden.co.uk/contact-us/
7,Freedom Health Insurance,https://www.freedomhealthinsurance.co.uk/,N/A,0800 999 2013,https://www.freedomhealthinsurance.co.uk/conta...
8,National Friendly,https://www.nationalfriendly.co.uk/,N/A,N/A,https://www.nationalfriendly.co.uk/
9,Saga Health Insurance,https://www.saga.co.uk/health-insurance,N/A,N/A,https://www.saga.co.uk/contact-us/insurance/he...


In [17]:
print(england_df[[
    "Name",
    "Coverage State",
    "Coverage Type",
    "Services Rendered",
    "Target Customers"
]].to_string(index=False))

                    Name Coverage State Coverage Type Services Rendered Target Customers
                    Bupa            N/A           N/A               N/A              N/A
              AXA Health            N/A           N/A               N/A              N/A
                   Aviva            N/A           N/A               N/A              N/A
                     WPA            N/A           N/A               N/A              N/A
          VitalityHealth            N/A           N/A               N/A              N/A
              The Exeter            N/A           N/A               N/A              N/A
         Benenden Health            N/A           N/A               N/A              N/A
Freedom Health Insurance            N/A           N/A               N/A              N/A
       National Friendly            N/A           N/A               N/A              N/A
   Saga Health Insurance            N/A           N/A               N/A              N/A
        Cigna Healthc

In [18]:
coverage_updates = {

    "Bupa": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Medical Insurance",
        "Services Rendered": "Private healthcare, health insurance, specialist treatment, hospital care, health assessments and wellbeing services",
        "Target Customers": "Individuals, families, employers and businesses"
    },

    "AXA Health": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Medical Insurance",
        "Services Rendered": "Private healthcare, hospital treatment, specialist consultations, diagnostic tests, mental health support and health assessments",
        "Target Customers": "Individuals, couples, families, self-employed people, small businesses and large corporates"
    },

    "Aviva": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Medical Insurance",
        "Services Rendered": "Private treatment, hospital care, specialist consultations, diagnostic tests, cancer treatment, mental health treatment and GP video consultations",
        "Target Customers": "Individuals, families, employees and businesses"
    },

    "WPA": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Medical Insurance / Health Cash Plans",
        "Services Rendered": "Private hospital treatment, consultations, diagnostic tests, health cash plans, dental plans and employee health services",
        "Target Customers": "Individuals, families, self-employed people and businesses"
    },

    "VitalityHealth": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Medical Insurance",
        "Services Rendered": "Private hospital treatment, GP services, diagnostic tests, specialist care, cancer care, mental health support, physiotherapy and wellbeing rewards",
        "Target Customers": "Individuals, families, self-employed people and businesses"
    },

    "The Exeter": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Medical Insurance",
        "Services Rendered": "Private medical treatment, specialist consultations, hospital care, diagnostics and healthcare support",
        "Target Customers": "Individuals, families and businesses"
    },

    "Benenden Health": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Healthcare Membership",
        "Services Rendered": "24/7 GP helpline, mental health support, diagnostic consultations and tests, surgical treatment, hospital access and health and wellbeing services",
        "Target Customers": "UK residents, individuals, families and members"
    },

    "Freedom Health Insurance": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Medical Insurance",
        "Services Rendered": "Private medical treatment, hospital care, specialist consultations, diagnostic services and healthcare support",
        "Target Customers": "Individuals, families, expatriates and businesses"
    },

    "National Friendly": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Medical Insurance",
        "Services Rendered": "Private healthcare, medical consultations, diagnostic services and treatment",
        "Target Customers": "Individuals and families"
    },

    "Saga Health Insurance": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Medical Insurance",
        "Services Rendered": "Private hospital treatment, inpatient and day-patient treatment, outpatient consultations, tests, scans, physiotherapy, cancer support and 24/7 GP services",
        "Target Customers": "People aged 50 and over"
    },

    "Cigna Healthcare": {
        "Coverage State": "United Kingdom and international",
        "Coverage Type": "International Private Medical Insurance",
        "Services Rendered": "International medical insurance, inpatient and day-patient care, hospital treatment, specialist care, outpatient care, dental options, telehealth and global healthcare support",
        "Target Customers": "Individuals, families, international students, workers, retirees, globally mobile employees and organisations"
    },

    "Simplyhealth": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Health Cash Plan / Health Insurance",
        "Services Rendered": "Dental care, optical care, physiotherapy, chiropractic care, podiatry, diagnostic consultations and scans, prescriptions and everyday healthcare costs",
        "Target Customers": "Individuals, families, employees and businesses"
    },

    "General & Medical": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Medical Insurance",
        "Services Rendered": "Private medical treatment, hospital care, specialist consultations, diagnostics and healthcare support",
        "Target Customers": "Individuals, families and businesses"
    },

    "CS Healthcare": {
        "Coverage State": "United Kingdom",
        "Coverage Type": "Private Healthcare / Health Insurance",
        "Services Rendered": "Private healthcare, consultations, diagnostic services, hospital treatment and healthcare support",
        "Target Customers": "Civil servants, public-sector employees, their families and eligible members"
    }
}

In [19]:
for company, info in coverage_updates.items():
    idx = england_df.index[england_df["Name"] == company]

    for column, value in info.items():
        england_df.loc[idx, column] = value

print("England enrichment complete.")

England enrichment complete.


In [20]:
england_df[
    [
        "Name",
        "Coverage State",
        "Coverage Type",
        "Services Rendered",
        "Target Customers"
    ]
]

,Name,Coverage State,Coverage Type,Services Rendered,Target Customers
0,Bupa,United Kingdom,Private Medical Insurance,"Private healthcare, health insurance, speciali...","Individuals, families, employers and businesses"
1,AXA Health,United Kingdom,Private Medical Insurance,"Private healthcare, hospital treatment, specia...","Individuals, couples, families, self-employed ..."
2,Aviva,United Kingdom,Private Medical Insurance,"Private treatment, hospital care, specialist c...","Individuals, families, employees and businesses"
3,WPA,United Kingdom,Private Medical Insurance / Health Cash Plans,"Private hospital treatment, consultations, dia...","Individuals, families, self-employed people an..."
4,VitalityHealth,United Kingdom,Private Medical Insurance,"Private hospital treatment, GP services, diagn...","Individuals, families, self-employed people an..."
5,The Exeter,United Kingdom,Private Medical Insurance,"Private medical treatment, specialist consulta...","Individuals, families and businesses"
6,Benenden Health,United Kingdom,Healthcare Membership,"24/7 GP helpline, mental health support, diagn...","UK residents, individuals, families and members"
7,Freedom Health Insurance,United Kingdom,Private Medical Insurance,"Private medical treatment, hospital care, spec...","Individuals, families, expatriates and businesses"
8,National Friendly,United Kingdom,Private Medical Insurance,"Private healthcare, medical consultations, dia...",Individuals and families
9,Saga Health Insurance,United Kingdom,Private Medical Insurance,"Private hospital treatment, inpatient and day-...",People aged 50 and over


In [24]:
england_df[
    [
        "Name",
        "State",
        "Website",
        "Email",
        "Contact Number",
        "Coverage Type",
        "Coverage State",
        "Services Rendered",
        "Target Customers",
        "Source URL"
    ]
]

,Name,State,Website,Email,Contact Number,Coverage Type,Coverage State,Services Rendered,Target Customers,Source URL
0,Bupa,England,https://www.bupa.co.uk/,N/A,0345 609 0111,Private Medical Insurance,United Kingdom,"Private healthcare, health insurance, speciali...","Individuals, families, employers and businesses",https://www.bupa.co.uk/contact-us.php
1,AXA Health,England,https://www.axahealth.co.uk/,N/A,0800 027 1384,Private Medical Insurance,United Kingdom,"Private healthcare, hospital treatment, specia...","Individuals, couples, families, self-employed ...",https://www.axahealth.co.uk/health-insurance/c...
2,Aviva,England,https://www.aviva.co.uk/,N/A,N/A,Private Medical Insurance,United Kingdom,"Private treatment, hospital care, specialist c...","Individuals, families, employees and businesses",https://www.aviva.co.uk/help-and-support/conta...
3,WPA,England,https://www.wpa.org.uk/,N/A,N/A,Private Medical Insurance / Health Cash Plans,United Kingdom,"Private hospital treatment, consultations, dia...","Individuals, families, self-employed people an...",https://www.wpa.org.uk/
4,VitalityHealth,England,https://www.vitality.co.uk/,N/A,N/A,Private Medical Insurance,United Kingdom,"Private hospital treatment, GP services, diagn...","Individuals, families, self-employed people an...",https://www.vitality.co.uk/
5,The Exeter,England,https://www.the-exeter.com/,member@the-exeter.com,0300 123 3201,Private Medical Insurance,United Kingdom,"Private medical treatment, specialist consulta...","Individuals, families and businesses",https://the-exeter.com/contact-us/member/
6,Benenden Health,England,https://www.benenden.co.uk/,N/A,N/A,Healthcare Membership,United Kingdom,"24/7 GP helpline, mental health support, diagn...","UK residents, individuals, families and members",https://www.benenden.co.uk/contact-us/
7,Freedom Health Insurance,England,https://www.freedomhealthinsurance.co.uk/,N/A,0800 999 2013,Private Medical Insurance,United Kingdom,"Private medical treatment, hospital care, spec...","Individuals, families, expatriates and businesses",https://www.freedomhealthinsurance.co.uk/conta...
8,National Friendly,England,https://www.nationalfriendly.co.uk/,N/A,N/A,Private Medical Insurance,United Kingdom,"Private healthcare, medical consultations, dia...",Individuals and families,https://www.nationalfriendly.co.uk/
9,Saga Health Insurance,England,https://www.saga.co.uk/health-insurance,N/A,N/A,Private Medical Insurance,United Kingdom,"Private hospital treatment, inpatient and day-...",People aged 50 and over,https://www.saga.co.uk/contact-us/insurance/he...


In [25]:
wales_df = england_df.copy()

In [26]:
%whos

Variable           Type          Data/Info
------------------------------------------
BeautifulSoup      type          <class 'bs4.BeautifulSoup'>
DDGS               _ProxyMeta    <class 'ddgs._DDGSProxy'>
column             str           Target Customers
columns            list          n=13
companies          list          n=14
company            str           CS Healthcare
contact_results    dict          n=14
contact_updates    dict          n=14
coverage_updates   dict          n=14
emails             list          n=0
england_df         DataFrame     Shape: (14, 13)
find_emails        function      <function find_emails at 0x0000020B80216020>
find_phones        function      <function find_phones at 0x0000020BFA9C6B60>
health_df          DataFrame     Shape: (14, 13)
i                  int           13
idx                Index         Index([13], dtype='int64')
info               dict          n=4
pd                 module        <module 'pandas' from 'C:<...>es\\pandas\\__init__

In [27]:
wales_df

,State,Region,Country,Name,Website,Email,Contact Number,Coverage State,Coverage Type,Services Rendered,Target Customers,Source,Source URL
0,England,N/A,United Kingdom,Bupa,https://www.bupa.co.uk/,N/A,0345 609 0111,United Kingdom,Private Medical Insurance,"Private healthcare, health insurance, speciali...","Individuals, families, employers and businesses",Official Company Website,https://www.bupa.co.uk/contact-us.php
1,England,N/A,United Kingdom,AXA Health,https://www.axahealth.co.uk/,N/A,0800 027 1384,United Kingdom,Private Medical Insurance,"Private healthcare, hospital treatment, specia...","Individuals, couples, families, self-employed ...",Official Company Website,https://www.axahealth.co.uk/health-insurance/c...
2,England,N/A,United Kingdom,Aviva,https://www.aviva.co.uk/,N/A,N/A,United Kingdom,Private Medical Insurance,"Private treatment, hospital care, specialist c...","Individuals, families, employees and businesses",Official Company Website,https://www.aviva.co.uk/help-and-support/conta...
3,England,N/A,United Kingdom,WPA,https://www.wpa.org.uk/,N/A,N/A,United Kingdom,Private Medical Insurance / Health Cash Plans,"Private hospital treatment, consultations, dia...","Individuals, families, self-employed people an...",Official Company Website,https://www.wpa.org.uk/
4,England,N/A,United Kingdom,VitalityHealth,https://www.vitality.co.uk/,N/A,N/A,United Kingdom,Private Medical Insurance,"Private hospital treatment, GP services, diagn...","Individuals, families, self-employed people an...",Official Company Website,https://www.vitality.co.uk/
5,England,N/A,United Kingdom,The Exeter,https://www.the-exeter.com/,member@the-exeter.com,0300 123 3201,United Kingdom,Private Medical Insurance,"Private medical treatment, specialist consulta...","Individuals, families and businesses",Official Company Website,https://the-exeter.com/contact-us/member/
6,England,N/A,United Kingdom,Benenden Health,https://www.benenden.co.uk/,N/A,N/A,United Kingdom,Healthcare Membership,"24/7 GP helpline, mental health support, diagn...","UK residents, individuals, families and members",Official Company Website,https://www.benenden.co.uk/contact-us/
7,England,N/A,United Kingdom,Freedom Health Insurance,https://www.freedomhealthinsurance.co.uk/,N/A,0800 999 2013,United Kingdom,Private Medical Insurance,"Private medical treatment, hospital care, spec...","Individuals, families, expatriates and businesses",Official Company Website,https://www.freedomhealthinsurance.co.uk/conta...
8,England,N/A,United Kingdom,National Friendly,https://www.nationalfriendly.co.uk/,N/A,N/A,United Kingdom,Private Medical Insurance,"Private healthcare, medical consultations, dia...",Individuals and families,Official Company Website,https://www.nationalfriendly.co.uk/
9,England,N/A,United Kingdom,Saga Health Insurance,https://www.saga.co.uk/health-insurance,N/A,N/A,United Kingdom,Private Medical Insurance,"Private hospital treatment, inpatient and day-...",People aged 50 and over,Official Company Website,https://www.saga.co.uk/contact-us/insurance/he...


In [28]:
wales_df = pd.DataFrame(columns=england_df.columns)

In [29]:
wales_df

,State,Region,Country,Name,Website,Email,Contact Number,Coverage State,Coverage Type,Services Rendered,Target Customers,Source,Source URL


In [30]:
wales_companies = [
    ["Aneurin Bevan University Health Board"],
    ["Betsi Cadwaladr University Health Board"],
    ["Cardiff and Vale University Health Board"],
    ["Cwm Taf Morgannwg University Health Board"],
    ["Hywel Dda University Health Board"],
    ["Powys Teaching Health Board"],
    ["Swansea Bay University Health Board"],
    ["Public Health Wales NHS Trust"],
    ["Velindre University NHS Trust"],
    ["Welsh Ambulance Services University NHS Trust"],
    ["Health Education and Improvement Wales"],
    ["Digital Health and Care Wales"]
]

wales_df = pd.DataFrame(wales_companies, columns=["Name"])

wales_df["State"] = "Wales"
wales_df["Region"] = "N/A"
wales_df["Country"] = "United Kingdom"

wales_df

,Name,State,Region,Country
0,Aneurin Bevan University Health Board,Wales,N/A,United Kingdom
1,Betsi Cadwaladr University Health Board,Wales,N/A,United Kingdom
2,Cardiff and Vale University Health Board,Wales,N/A,United Kingdom
3,Cwm Taf Morgannwg University Health Board,Wales,N/A,United Kingdom
4,Hywel Dda University Health Board,Wales,N/A,United Kingdom
5,Powys Teaching Health Board,Wales,N/A,United Kingdom
6,Swansea Bay University Health Board,Wales,N/A,United Kingdom
7,Public Health Wales NHS Trust,Wales,N/A,United Kingdom
8,Velindre University NHS Trust,Wales,N/A,United Kingdom
9,Welsh Ambulance Services University NHS Trust,Wales,N/A,United Kingdom


In [31]:
wales_df["Website"] = "N/A"
wales_df["Email"] = "N/A"
wales_df["Contact Number"] = "N/A"
wales_df["Coverage Type"] = "N/A"
wales_df["Coverage State"] = "N/A"
wales_df["Services Rendered"] = "N/A"
wales_df["Target Customers"] = "N/A"
wales_df["Source"] = "Official Organisation Website"
wales_df["Source URL"] = "N/A"

wales_df

,Name,State,Region,Country,Website,Email,Contact Number,Coverage Type,Coverage State,Services Rendered,Target Customers,Source,Source URL
0,Aneurin Bevan University Health Board,Wales,N/A,United Kingdom,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Official Organisation Website,N/A
1,Betsi Cadwaladr University Health Board,Wales,N/A,United Kingdom,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Official Organisation Website,N/A
2,Cardiff and Vale University Health Board,Wales,N/A,United Kingdom,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Official Organisation Website,N/A
3,Cwm Taf Morgannwg University Health Board,Wales,N/A,United Kingdom,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Official Organisation Website,N/A
4,Hywel Dda University Health Board,Wales,N/A,United Kingdom,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Official Organisation Website,N/A
5,Powys Teaching Health Board,Wales,N/A,United Kingdom,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Official Organisation Website,N/A
6,Swansea Bay University Health Board,Wales,N/A,United Kingdom,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Official Organisation Website,N/A
7,Public Health Wales NHS Trust,Wales,N/A,United Kingdom,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Official Organisation Website,N/A
8,Velindre University NHS Trust,Wales,N/A,United Kingdom,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Official Organisation Website,N/A
9,Welsh Ambulance Services University NHS Trust,Wales,N/A,United Kingdom,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Official Organisation Website,N/A


In [32]:
wales_df.columns

Index(['Name', 'State', 'Region', 'Country', 'Website', 'Email',
       'Contact Number', 'Coverage Type', 'Coverage State',
       'Services Rendered', 'Target Customers', 'Source', 'Source URL'],
      dtype='object')

In [33]:
wales_websites = {
    "Aneurin Bevan University Health Board": "https://abuhb.nhs.wales/",
    "Betsi Cadwaladr University Health Board": "https://bcuhb.nhs.wales/",
    "Cardiff and Vale University Health Board": "https://cavuhb.nhs.wales/",
    "Cwm Taf Morgannwg University Health Board": "https://ctmuhb.nhs.wales/",
    "Hywel Dda University Health Board": "https://hduhb.nhs.wales/",
    "Powys Teaching Health Board": "https://pthb.nhs.wales/",
    "Swansea Bay University Health Board": "https://sbuhb.nhs.wales/",
    "Public Health Wales NHS Trust": "https://phw.nhs.wales/",
    "Velindre University NHS Trust": "https://velindre.nhs.wales/",
    "Welsh Ambulance Services University NHS Trust": "https://ambulance.nhs.wales/",
    "Health Education and Improvement Wales": "https://heiw.nhs.wales/",
    "Digital Health and Care Wales": "https://dhcw.nhs.wales/"
}

wales_df["Website"] = wales_df["Name"].map(wales_websites)

wales_df[["Name", "Website"]]

,Name,Website
0,Aneurin Bevan University Health Board,https://abuhb.nhs.wales/
1,Betsi Cadwaladr University Health Board,https://bcuhb.nhs.wales/
2,Cardiff and Vale University Health Board,https://cavuhb.nhs.wales/
3,Cwm Taf Morgannwg University Health Board,https://ctmuhb.nhs.wales/
4,Hywel Dda University Health Board,https://hduhb.nhs.wales/
5,Powys Teaching Health Board,https://pthb.nhs.wales/
6,Swansea Bay University Health Board,https://sbuhb.nhs.wales/
7,Public Health Wales NHS Trust,https://phw.nhs.wales/
8,Velindre University NHS Trust,https://velindre.nhs.wales/
9,Welsh Ambulance Services University NHS Trust,https://ambulance.nhs.wales/


In [35]:
wales_df["Website"] = wales_df["Name"].map(wales_websites)

In [36]:
wales_contacts = {
    "Aneurin Bevan University Health Board": {
        "Email": "abb.engagement@wales.nhs.uk",
        "Contact Number": "01633 431890",
        "Source URL": "https://abuhb.nhs.wales/about-us/engagement/community-engagement/contact-us/"
    },

    "Betsi Cadwaladr University Health Board": {
        "Email": "N/A",
        "Contact Number": "N/A",
        "Source URL": "https://bcuhb.nhs.wales/contact-us/"
    },

    "Cardiff and Vale University Health Board": {
        "Email": "Cav.Concerns@wales.nhs.uk",
        "Contact Number": "029 2183 6318",
        "Source URL": "https://cavuhb.nhs.wales/contact-us/"
    },

    "Cwm Taf Morgannwg University Health Board": {
        "Email": "N/A",
        "Contact Number": "N/A",
        "Source URL": "https://ctmuhb.nhs.wales/contact-us/"
    },

    "Hywel Dda University Health Board": {
        "Email": "corporate.correspondence.hdd@wales.nhs.uk",
        "Contact Number": "01267 235151",
        "Source URL": "https://hduhb.nhs.wales/about-us/your-health-board/board-members/our-board-members/contact-board-members/"
    },

    "Powys Teaching Health Board": {
        "Email": "N/A",
        "Contact Number": "01874 442071",
        "Source URL": "https://pthb.nhs.wales/use-of-site/freedom-of-information/"
    },

    "Swansea Bay University Health Board": {
        "Email": "FOIA.Requests@wales.nhs.uk",
        "Contact Number": "01639 683344",
        "Source URL": "https://sbuhb.nhs.wales/about-us/contact-us/"
    },

    "Public Health Wales NHS Trust": {
        "Email": "N/A",
        "Contact Number": "029 2022 7744",
        "Source URL": "https://phw.nhs.wales/contact-us/"
    },

    "Velindre University NHS Trust": {
        "Email": "N/A",
        "Contact Number": "N/A",
        "Source URL": "https://velindre.nhs.wales/contact-us/"
    },

    "Welsh Ambulance Services University NHS Trust": {
        "Email": "PECI.team@wales.nhs.uk",
        "Contact Number": "0300 723 9207",
        "Source URL": "https://ambulance.nhs.wales/contact-us/"
    },

    "Health Education and Improvement Wales": {
        "Email": "N/A",
        "Contact Number": "N/A",
        "Source URL": "https://heiw.nhs.wales/contact-us/"
    },

    "Digital Health and Care Wales": {
        "Email": "DHCW-Comms@wales.nhs.uk",
        "Contact Number": "02920 500500",
        "Source URL": "https://dhcw.nhs.wales/contact-us/"
    }
}

In [37]:
for company, info in wales_contacts.items():
    idx = wales_df.index[wales_df["Name"] == company]

    if len(idx) > 0:
        wales_df.loc[idx, "Email"] = info["Email"]
        wales_df.loc[idx, "Contact Number"] = info["Contact Number"]
        wales_df.loc[idx, "Source URL"] = info["Source URL"]

wales_df[["Name", "Email", "Contact Number", "Source URL"]]

,Name,Email,Contact Number,Source URL
0,Aneurin Bevan University Health Board,abb.engagement@wales.nhs.uk,01633 431890,https://abuhb.nhs.wales/about-us/engagement/co...
1,Betsi Cadwaladr University Health Board,N/A,N/A,https://bcuhb.nhs.wales/contact-us/
2,Cardiff and Vale University Health Board,Cav.Concerns@wales.nhs.uk,029 2183 6318,https://cavuhb.nhs.wales/contact-us/
3,Cwm Taf Morgannwg University Health Board,N/A,N/A,https://ctmuhb.nhs.wales/contact-us/
4,Hywel Dda University Health Board,corporate.correspondence.hdd@wales.nhs.uk,01267 235151,https://hduhb.nhs.wales/about-us/your-health-b...
5,Powys Teaching Health Board,N/A,01874 442071,https://pthb.nhs.wales/use-of-site/freedom-of-...
6,Swansea Bay University Health Board,FOIA.Requests@wales.nhs.uk,01639 683344,https://sbuhb.nhs.wales/about-us/contact-us/
7,Public Health Wales NHS Trust,N/A,029 2022 7744,https://phw.nhs.wales/contact-us/
8,Velindre University NHS Trust,N/A,N/A,https://velindre.nhs.wales/contact-us/
9,Welsh Ambulance Services University NHS Trust,PECI.team@wales.nhs.uk,0300 723 9207,https://ambulance.nhs.wales/contact-us/


In [38]:
wales_df[["Name", "Email", "Contact Number"]]

,Name,Email,Contact Number
0,Aneurin Bevan University Health Board,abb.engagement@wales.nhs.uk,01633 431890
1,Betsi Cadwaladr University Health Board,N/A,N/A
2,Cardiff and Vale University Health Board,Cav.Concerns@wales.nhs.uk,029 2183 6318
3,Cwm Taf Morgannwg University Health Board,N/A,N/A
4,Hywel Dda University Health Board,corporate.correspondence.hdd@wales.nhs.uk,01267 235151
5,Powys Teaching Health Board,N/A,01874 442071
6,Swansea Bay University Health Board,FOIA.Requests@wales.nhs.uk,01639 683344
7,Public Health Wales NHS Trust,N/A,029 2022 7744
8,Velindre University NHS Trust,N/A,N/A
9,Welsh Ambulance Services University NHS Trust,PECI.team@wales.nhs.uk,0300 723 9207


In [39]:
coverage_types = {
    "Aneurin Bevan University Health Board": "NHS Health Services",
    "Betsi Cadwaladr University Health Board": "NHS Health Services",
    "Cardiff and Vale University Health Board": "NHS Health Services",
    "Cwm Taf Morgannwg University Health Board": "NHS Health Services",
    "Hywel Dda University Health Board": "NHS Health Services",
    "Powys Teaching Health Board": "NHS Health Services",
    "Swansea Bay University Health Board": "NHS Health Services",
    
    "Public Health Wales NHS Trust": "Public Health Services",
    
    "Velindre University NHS Trust": "Specialist NHS Healthcare",
    
    "Welsh Ambulance Services University NHS Trust": "Emergency and Non-Emergency Ambulance Services",
    
    "Health Education and Improvement Wales": "NHS Workforce Education and Training",
    
    "Digital Health and Care Wales": "Digital Health and Data Services"
}

wales_df["Coverage Type"] = wales_df["Name"].map(coverage_types)

wales_df[["Name", "Coverage Type"]]

,Name,Coverage Type
0,Aneurin Bevan University Health Board,NHS Health Services
1,Betsi Cadwaladr University Health Board,NHS Health Services
2,Cardiff and Vale University Health Board,NHS Health Services
3,Cwm Taf Morgannwg University Health Board,NHS Health Services
4,Hywel Dda University Health Board,NHS Health Services
5,Powys Teaching Health Board,NHS Health Services
6,Swansea Bay University Health Board,NHS Health Services
7,Public Health Wales NHS Trust,Public Health Services
8,Velindre University NHS Trust,Specialist NHS Healthcare
9,Welsh Ambulance Services University NHS Trust,Emergency and Non-Emergency Ambulance Services


In [40]:
coverage_state = {
    "Aneurin Bevan University Health Board": "Local – Southeast Wales",
    "Betsi Cadwaladr University Health Board": "Local – North Wales",
    "Cardiff and Vale University Health Board": "Local – Cardiff and Vale of Glamorgan",
    "Cwm Taf Morgannwg University Health Board": "Local – Cwm Taf Morgannwg",
    "Hywel Dda University Health Board": "Local – West Wales",
    "Powys Teaching Health Board": "Local – Powys",
    "Swansea Bay University Health Board": "Local – Swansea Bay",
    "Public Health Wales NHS Trust": "National – Wales",
    "Velindre University NHS Trust": "National – Wales",
    "Welsh Ambulance Services University NHS Trust": "National – Wales",
    "Health Education and Improvement Wales": "National – Wales",
    "Digital Health and Care Wales": "National – Wales"
}

wales_df["Coverage State"] = wales_df["Name"].map(coverage_state)

wales_df[["Name", "Coverage State"]]

,Name,Coverage State
0,Aneurin Bevan University Health Board,Local – Southeast Wales
1,Betsi Cadwaladr University Health Board,Local – North Wales
2,Cardiff and Vale University Health Board,Local – Cardiff and Vale of Glamorgan
3,Cwm Taf Morgannwg University Health Board,Local – Cwm Taf Morgannwg
4,Hywel Dda University Health Board,Local – West Wales
5,Powys Teaching Health Board,Local – Powys
6,Swansea Bay University Health Board,Local – Swansea Bay
7,Public Health Wales NHS Trust,National – Wales
8,Velindre University NHS Trust,National – Wales
9,Welsh Ambulance Services University NHS Trust,National – Wales


In [41]:
services_rendered = {
    "Aneurin Bevan University Health Board":
        "Primary care, hospital care, community health services, mental health services, specialist healthcare and public health services",

    "Betsi Cadwaladr University Health Board":
        "Primary care, hospital care, community health services, mental health services, specialist healthcare and emergency care",

    "Cardiff and Vale University Health Board":
        "Primary care, hospital care, community health services, mental health services, specialist healthcare and emergency care",

    "Cwm Taf Morgannwg University Health Board":
        "Primary care, hospital care, community health services, mental health services, specialist healthcare and emergency care",

    "Hywel Dda University Health Board":
        "Primary care, hospital care, community health services, mental health services, cancer care, diagnostics, physiotherapy and specialist healthcare",

    "Powys Teaching Health Board":
        "Primary care, community hospitals, community health services, mental health services, therapy services and specialist care through commissioned services",

    "Swansea Bay University Health Board":
        "Hospital care, primary care, community health services, mental health and learning disability services, specialist healthcare and emergency care",

    "Public Health Wales NHS Trust":
        "Screening programmes, immunisations and vaccines, health data and surveillance, STI testing, disease control, workplace health and public health programmes",

    "Velindre University NHS Trust":
        "Specialist cancer treatment, radiotherapy, systemic anti-cancer treatments, specialist palliative care, cancer research and Welsh Blood Service",

    "Welsh Ambulance Services University NHS Trust":
        "Emergency ambulance services, non-emergency patient transport, emergency medical services and NHS 111 Wales",

    "Health Education and Improvement Wales":
        "Workforce education, training, professional development, workforce planning and development for NHS Wales",

    "Digital Health and Care Wales":
        "National digital health systems, digital and data services, electronic health records, digital infrastructure, cyber security and digital transformation"
}

wales_df["Services Rendered"] = wales_df["Name"].map(services_rendered)

wales_df[["Name", "Services Rendered"]]

,Name,Services Rendered
0,Aneurin Bevan University Health Board,"Primary care, hospital care, community health ..."
1,Betsi Cadwaladr University Health Board,"Primary care, hospital care, community health ..."
2,Cardiff and Vale University Health Board,"Primary care, hospital care, community health ..."
3,Cwm Taf Morgannwg University Health Board,"Primary care, hospital care, community health ..."
4,Hywel Dda University Health Board,"Primary care, hospital care, community health ..."
5,Powys Teaching Health Board,"Primary care, community hospitals, community h..."
6,Swansea Bay University Health Board,"Hospital care, primary care, community health ..."
7,Public Health Wales NHS Trust,"Screening programmes, immunisations and vaccin..."
8,Velindre University NHS Trust,"Specialist cancer treatment, radiotherapy, sys..."
9,Welsh Ambulance Services University NHS Trust,"Emergency ambulance services, non-emergency pa..."


In [42]:
target_customers = {
    "Aneurin Bevan University Health Board": "Residents of Blaenau Gwent, Caerphilly, Monmouthshire, Newport and Torfaen",
    "Betsi Cadwaladr University Health Board": "Residents of North Wales",
    "Cardiff and Vale University Health Board": "Residents of Cardiff and the Vale of Glamorgan",
    "Cwm Taf Morgannwg University Health Board": "Residents of Bridgend, Merthyr Tydfil and Rhondda Cynon Taf",
    "Hywel Dda University Health Board": "Residents of Carmarthenshire, Ceredigion, Pembrokeshire and parts of South Powys",
    "Powys Teaching Health Board": "Residents of Powys",
    "Swansea Bay University Health Board": "Residents of Neath Port Talbot and Swansea",
    "Public Health Wales NHS Trust": "People across Wales, including individuals, communities and public-sector organisations",
    "Velindre University NHS Trust": "Patients across Wales requiring specialist cancer and related services",
    "Welsh Ambulance Services University NHS Trust": "People across Wales requiring emergency or non-emergency ambulance services",
    "Health Education and Improvement Wales": "Healthcare professionals, healthcare students and the NHS workforce in Wales",
    "Digital Health and Care Wales": "NHS Wales organisations, healthcare professionals and the people who use NHS Wales services"
}

wales_df["Target Customers"] = wales_df["Name"].map(target_customers)

wales_df[["Name", "Target Customers"]]

,Name,Target Customers
0,Aneurin Bevan University Health Board,"Residents of Blaenau Gwent, Caerphilly, Monmou..."
1,Betsi Cadwaladr University Health Board,Residents of North Wales
2,Cardiff and Vale University Health Board,Residents of Cardiff and the Vale of Glamorgan
3,Cwm Taf Morgannwg University Health Board,"Residents of Bridgend, Merthyr Tydfil and Rhon..."
4,Hywel Dda University Health Board,"Residents of Carmarthenshire, Ceredigion, Pemb..."
5,Powys Teaching Health Board,Residents of Powys
6,Swansea Bay University Health Board,Residents of Neath Port Talbot and Swansea
7,Public Health Wales NHS Trust,"People across Wales, including individuals, co..."
8,Velindre University NHS Trust,Patients across Wales requiring specialist can...
9,Welsh Ambulance Services University NHS Trust,People across Wales requiring emergency or non...


In [43]:
print(england_df.columns.tolist())
print(wales_df.columns.tolist())

['State', 'Region', 'Country', 'Name', 'Website', 'Email', 'Contact Number', 'Coverage State', 'Coverage Type', 'Services Rendered', 'Target Customers', 'Source', 'Source URL']
['Name', 'State', 'Region', 'Country', 'Website', 'Email', 'Contact Number', 'Coverage Type', 'Coverage State', 'Services Rendered', 'Target Customers', 'Source', 'Source URL']


In [44]:
england_wales_df = pd.concat(
    [england_df, wales_df],
    ignore_index=True
)

In [45]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", None)

display(england_wales_df)

,State,Region,Country,Name,Website,Email,Contact Number,Coverage State,Coverage Type,Services Rendered,Target Customers,Source,Source URL
0,England,N/A,United Kingdom,Bupa,https://www.bupa.co.uk/,N/A,0345 609 0111,United Kingdom,Private Medical Insurance,"Private healthcare, health insurance, specialist treatment, hospital care, health assessments an...","Individuals, families, employers and businesses",Official Company Website,https://www.bupa.co.uk/contact-us.php
1,England,N/A,United Kingdom,AXA Health,https://www.axahealth.co.uk/,N/A,0800 027 1384,United Kingdom,Private Medical Insurance,"Private healthcare, hospital treatment, specialist consultations, diagnostic tests, mental healt...","Individuals, couples, families, self-employed people, small businesses and large corporates",Official Company Website,https://www.axahealth.co.uk/health-insurance/contact-us/
2,England,N/A,United Kingdom,Aviva,https://www.aviva.co.uk/,N/A,N/A,United Kingdom,Private Medical Insurance,"Private treatment, hospital care, specialist consultations, diagnostic tests, cancer treatment, ...","Individuals, families, employees and businesses",Official Company Website,https://www.aviva.co.uk/help-and-support/contact-us/
3,England,N/A,United Kingdom,WPA,https://www.wpa.org.uk/,N/A,N/A,United Kingdom,Private Medical Insurance / Health Cash Plans,"Private hospital treatment, consultations, diagnostic tests, health cash plans, dental plans and...","Individuals, families, self-employed people and businesses",Official Company Website,https://www.wpa.org.uk/
4,England,N/A,United Kingdom,VitalityHealth,https://www.vitality.co.uk/,N/A,N/A,United Kingdom,Private Medical Insurance,"Private hospital treatment, GP services, diagnostic tests, specialist care, cancer care, mental ...","Individuals, families, self-employed people and businesses",Official Company Website,https://www.vitality.co.uk/
5,England,N/A,United Kingdom,The Exeter,https://www.the-exeter.com/,member@the-exeter.com,0300 123 3201,United Kingdom,Private Medical Insurance,"Private medical treatment, specialist consultations, hospital care, diagnostics and healthcare s...","Individuals, families and businesses",Official Company Website,https://the-exeter.com/contact-us/member/
6,England,N/A,United Kingdom,Benenden Health,https://www.benenden.co.uk/,N/A,N/A,United Kingdom,Healthcare Membership,"24/7 GP helpline, mental health support, diagnostic consultations and tests, surgical treatment,...","UK residents, individuals, families and members",Official Company Website,https://www.benenden.co.uk/contact-us/
7,England,N/A,United Kingdom,Freedom Health Insurance,https://www.freedomhealthinsurance.co.uk/,N/A,0800 999 2013,United Kingdom,Private Medical Insurance,"Private medical treatment, hospital care, specialist consultations, diagnostic services and heal...","Individuals, families, expatriates and businesses",Official Company Website,https://www.freedomhealthinsurance.co.uk/contact-us
8,England,N/A,United Kingdom,National Friendly,https://www.nationalfriendly.co.uk/,N/A,N/A,United Kingdom,Private Medical Insurance,"Private healthcare, medical consultations, diagnostic services and treatment",Individuals and families,Official Company Website,https://www.nationalfriendly.co.uk/
9,England,N/A,United Kingdom,Saga Health Insurance,https://www.saga.co.uk/health-insurance,N/A,N/A,United Kingdom,Private Medical Insurance,"Private hospital treatment, inpatient and day-patient treatment, outpatient consultations, tests...",People aged 50 and over,Official Company Website,https://www.saga.co.uk/contact-us/insurance/health-insurance


In [46]:
print("Total organisations:", len(england_wales_df))
print("\nBy State:")
print(england_wales_df["State"].value_counts())

Total organisations: 26

By State:
State
England    14
Wales      12
Name: count, dtype: int64


In [47]:
display(england_wales_df)

,State,Region,Country,Name,Website,Email,Contact Number,Coverage State,Coverage Type,Services Rendered,Target Customers,Source,Source URL
0,England,N/A,United Kingdom,Bupa,https://www.bupa.co.uk/,N/A,0345 609 0111,United Kingdom,Private Medical Insurance,"Private healthcare, health insurance, specialist treatment, hospital care, health assessments an...","Individuals, families, employers and businesses",Official Company Website,https://www.bupa.co.uk/contact-us.php
1,England,N/A,United Kingdom,AXA Health,https://www.axahealth.co.uk/,N/A,0800 027 1384,United Kingdom,Private Medical Insurance,"Private healthcare, hospital treatment, specialist consultations, diagnostic tests, mental healt...","Individuals, couples, families, self-employed people, small businesses and large corporates",Official Company Website,https://www.axahealth.co.uk/health-insurance/contact-us/
2,England,N/A,United Kingdom,Aviva,https://www.aviva.co.uk/,N/A,N/A,United Kingdom,Private Medical Insurance,"Private treatment, hospital care, specialist consultations, diagnostic tests, cancer treatment, ...","Individuals, families, employees and businesses",Official Company Website,https://www.aviva.co.uk/help-and-support/contact-us/
3,England,N/A,United Kingdom,WPA,https://www.wpa.org.uk/,N/A,N/A,United Kingdom,Private Medical Insurance / Health Cash Plans,"Private hospital treatment, consultations, diagnostic tests, health cash plans, dental plans and...","Individuals, families, self-employed people and businesses",Official Company Website,https://www.wpa.org.uk/
4,England,N/A,United Kingdom,VitalityHealth,https://www.vitality.co.uk/,N/A,N/A,United Kingdom,Private Medical Insurance,"Private hospital treatment, GP services, diagnostic tests, specialist care, cancer care, mental ...","Individuals, families, self-employed people and businesses",Official Company Website,https://www.vitality.co.uk/
5,England,N/A,United Kingdom,The Exeter,https://www.the-exeter.com/,member@the-exeter.com,0300 123 3201,United Kingdom,Private Medical Insurance,"Private medical treatment, specialist consultations, hospital care, diagnostics and healthcare s...","Individuals, families and businesses",Official Company Website,https://the-exeter.com/contact-us/member/
6,England,N/A,United Kingdom,Benenden Health,https://www.benenden.co.uk/,N/A,N/A,United Kingdom,Healthcare Membership,"24/7 GP helpline, mental health support, diagnostic consultations and tests, surgical treatment,...","UK residents, individuals, families and members",Official Company Website,https://www.benenden.co.uk/contact-us/
7,England,N/A,United Kingdom,Freedom Health Insurance,https://www.freedomhealthinsurance.co.uk/,N/A,0800 999 2013,United Kingdom,Private Medical Insurance,"Private medical treatment, hospital care, specialist consultations, diagnostic services and heal...","Individuals, families, expatriates and businesses",Official Company Website,https://www.freedomhealthinsurance.co.uk/contact-us
8,England,N/A,United Kingdom,National Friendly,https://www.nationalfriendly.co.uk/,N/A,N/A,United Kingdom,Private Medical Insurance,"Private healthcare, medical consultations, diagnostic services and treatment",Individuals and families,Official Company Website,https://www.nationalfriendly.co.uk/
9,England,N/A,United Kingdom,Saga Health Insurance,https://www.saga.co.uk/health-insurance,N/A,N/A,United Kingdom,Private Medical Insurance,"Private hospital treatment, inpatient and day-patient treatment, outpatient consultations, tests...",People aged 50 and over,Official Company Website,https://www.saga.co.uk/contact-us/insurance/health-insurance


In [48]:
england_wales_df.to_excel(
    "England_Wales_Healthcare_Organisations.xlsx",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [49]:
england_wales_df.shape

(26, 13)

In [50]:
nigeria_df = pd.DataFrame(columns=[
    "State",
    "Region",
    "Country",
    "Name",
    "Website",
    "Email",
    "Contact Number",
    "Coverage State",
    "Coverage Type",
    "Services Rendered",
    "Target Customers",
    "Source",
    "Source URL"
])

nigeria_df

,State,Region,Country,Name,Website,Email,Contact Number,Coverage State,Coverage Type,Services Rendered,Target Customers,Source,Source URL


In [52]:
import requests
import pandas as pd
from io import StringIO

nhia_url = "https://www.nhia.gov.ng/hmo/"

response = requests.get(nhia_url, verify=False, timeout=30)

print(response.status_code)

tables = pd.read_html(StringIO(response.text))

print("Number of tables found:", len(tables))

for i, table in enumerate(tables):
    print("Table", i, table.shape)
    display(table.head())

C:\Users\user\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.nhia.gov.ng'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200
Number of tables found: 1
Table 0 (94, 7)


,S/NO.,HMO,HMO ID,WEBSITES,Address,Email,Call Center Numbers
0,1,A&M HEALTHCARE TRUST LIMITED,102,Click here,"Plot U Bekaji Road, Opposite Bekaji Jumma'at Mosque, Jimeta, Yola, Adamawa State",info@amhmo.com,"08033646497, 09162788582, 08024143666"
1,2,AIICO MULTISHIELD NIGERIA LIMITED,6,Click here,"322, IKORODU ROAD, ANTHONY",info@aiicomultishield.com loshunniyi@aiicomultishield.com,O7026744353 O8056744353
2,3,ALLEANZA HEALTH MANAGEMENT LIMITED,111,Click here,"? 83B, Basheer Shittu Avenue, Magodo Estate, Phase 2, Shangisha, Lagos",info@alleanzahealth.com,O7036592835
3,4,ALLY HEALTHCARE LIMITED,119,NaN,"Suit C4 Plot 1196 Ndjamena Crescent, Wuse II, Abuja",allyhealthcarelimited@gmail.com,O8148808171
4,5,AMAN HEALTH MAINTENANCE ORGANIZATIONS,121,Click here,"? 1, M. M. Alkali Street, Off 442 Crescent, Citec Vilas Gwarinpa, Abuja",info@amanhmo.com,O8100588906


In [53]:
nigeria_hmo = tables[0].copy()

nigeria_hmo

,S/NO.,HMO,HMO ID,WEBSITES,Address,Email,Call Center Numbers
0,1,A&M HEALTHCARE TRUST LIMITED,102,Click here,"Plot U Bekaji Road, Opposite Bekaji Jumma'at Mosque, Jimeta, Yola, Adamawa State",info@amhmo.com,"08033646497, 09162788582, 08024143666"
1,2,AIICO MULTISHIELD NIGERIA LIMITED,6,Click here,"322, IKORODU ROAD, ANTHONY",info@aiicomultishield.com loshunniyi@aiicomultishield.com,O7026744353 O8056744353
2,3,ALLEANZA HEALTH MANAGEMENT LIMITED,111,Click here,"? 83B, Basheer Shittu Avenue, Magodo Estate, Phase 2, Shangisha, Lagos",info@alleanzahealth.com,O7036592835
3,4,ALLY HEALTHCARE LIMITED,119,NaN,"Suit C4 Plot 1196 Ndjamena Crescent, Wuse II, Abuja",allyhealthcarelimited@gmail.com,O8148808171
4,5,AMAN HEALTH MAINTENANCE ORGANIZATIONS,121,Click here,"? 1, M. M. Alkali Street, Off 442 Crescent, Citec Vilas Gwarinpa, Abuja",info@amanhmo.com,O8100588906
5,6,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,76,Click here,GOLDCREST MALL BUILDING,info@anchorhmo.com,"O8031230306, O7058890062, O7080601192"
6,7,ASHMED INTEGRATED HEALTH SERVICES LTD.,89,Click here,"RE 013 ZUNGERU CLOSE BY BIMA ROAD NITEL QUARTERS, KADUNA",info@ashmedintegratedhealthservices.com,O7036035182
7,8,AVON HEALTHCARE LIMITED,63,Click here,"22B, GLOVER ROAD",info@avonhealthcare.com,"O8102659972, O7002779800"
8,9,AXA MANSARD HEALTH LIMITED,59,Click here,"177, IKORODU ROAD, ONIPANU, LAGOS",healthcare@axamansard.com,O700AXAMANSARD
9,10,BASTION HEALTH LIMITED,97,Click here,"UBN HOUSE NO 1 ADEOLA ODEKU STREET, VICTORIA ISLAND, LAGOS, ETI OSA, LAGOS, Nigeria",Naomiduku@bastionhmo.com,0802BASTIONHMO 08002278466


In [54]:
nigeria_hmo.columns = [
    "S/NO.",
    "Name",
    "HMO ID",
    "Website",
    "Address",
    "Email",
    "Contact Number"
]

nigeria_hmo.head()

,S/NO.,Name,HMO ID,Website,Address,Email,Contact Number
0,1,A&M HEALTHCARE TRUST LIMITED,102,Click here,"Plot U Bekaji Road, Opposite Bekaji Jumma'at Mosque, Jimeta, Yola, Adamawa State",info@amhmo.com,"08033646497, 09162788582, 08024143666"
1,2,AIICO MULTISHIELD NIGERIA LIMITED,6,Click here,"322, IKORODU ROAD, ANTHONY",info@aiicomultishield.com loshunniyi@aiicomultishield.com,O7026744353 O8056744353
2,3,ALLEANZA HEALTH MANAGEMENT LIMITED,111,Click here,"? 83B, Basheer Shittu Avenue, Magodo Estate, Phase 2, Shangisha, Lagos",info@alleanzahealth.com,O7036592835
3,4,ALLY HEALTHCARE LIMITED,119,NaN,"Suit C4 Plot 1196 Ndjamena Crescent, Wuse II, Abuja",allyhealthcarelimited@gmail.com,O8148808171
4,5,AMAN HEALTH MAINTENANCE ORGANIZATIONS,121,Click here,"? 1, M. M. Alkali Street, Off 442 Crescent, Citec Vilas Gwarinpa, Abuja",info@amanhmo.com,O8100588906


In [55]:
nigeria_hmo["Country"] = "Nigeria"
nigeria_hmo["Source"] = "NHIA"
nigeria_hmo["Source URL"] = nhia_url

In [56]:
nigeria_hmo["State"] = "N/A"
nigeria_hmo["Region"] = "N/A"
nigeria_hmo["Coverage State"] = "N/A"
nigeria_hmo["Coverage Type"] = "N/A"
nigeria_hmo["Services Rendered"] = "N/A"
nigeria_hmo["Target Customers"] = "N/A"

In [57]:
import requests
from bs4 import BeautifulSoup

response = requests.get(nhia_url, verify=False, timeout=30)

soup = BeautifulSoup(response.text, "html.parser")

links = []

for a in soup.find_all("a"):
    text = a.get_text(strip=True)
    href = a.get("href")

    if text.lower() == "click here" and href:
        links.append(href)

print("Number of website links found:", len(links))

C:\Users\user\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.nhia.gov.ng'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Number of website links found: 88


In [58]:
links[:10]

['https://amhmo.com/',
 'https://www.aiicomultishield.com/',
 'https://alleanzahealth.com/individual-plans/',
 'https://amanhmo.com/home',
 'https://anchorhmo.com/',
 'https://ashmedintegratedhealthservices.com/',
 'https://www.avonhealthcare.com/',
 'https://www.axamansard.com/health/plans/',
 'https://bastionhmo.com/',
 'https://www.centurymedicaid.org/']

In [59]:
nigeria_hmo["Website"] = "N/A"

website_indices = nigeria_hmo.index[:len(links)]

nigeria_hmo.loc[website_indices, "Website"] = links

nigeria_hmo[["Name", "Website"]]

,Name,Website
0,A&M HEALTHCARE TRUST LIMITED,https://amhmo.com/
1,AIICO MULTISHIELD NIGERIA LIMITED,https://www.aiicomultishield.com/
2,ALLEANZA HEALTH MANAGEMENT LIMITED,https://alleanzahealth.com/individual-plans/
3,ALLY HEALTHCARE LIMITED,https://amanhmo.com/home
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,https://anchorhmo.com/
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,https://ashmedintegratedhealthservices.com/
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,https://www.avonhealthcare.com/
7,AVON HEALTHCARE LIMITED,https://www.axamansard.com/health/plans/
8,AXA MANSARD HEALTH LIMITED,https://bastionhmo.com/
9,BASTION HEALTH LIMITED,https://www.centurymedicaid.org/


In [60]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

response = requests.get(nhia_url, verify=False, timeout=30)
soup = BeautifulSoup(response.text, "html.parser")

rows = []

table = soup.find("table")

for tr in table.find_all("tr"):
    cells = tr.find_all(["td", "th"])

    if len(cells) >= 7:
        hmo_name = cells[1].get_text(" ", strip=True)

        link = cells[3].find("a")
        
        if link and link.get("href"):
            website = link.get("href")
        else:
            website = "N/A"

        rows.append([hmo_name, website])

website_df = pd.DataFrame(rows, columns=["Name", "Website"])

website_df

C:\Users\user\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.nhia.gov.ng'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,Name,Website
0,HMO,N/A
1,A&M HEALTHCARE TRUST LIMITED,https://amhmo.com/
2,AIICO MULTISHIELD NIGERIA LIMITED,https://www.aiicomultishield.com/
3,ALLEANZA HEALTH MANAGEMENT LIMITED,https://alleanzahealth.com/individual-plans/
4,ALLY HEALTHCARE LIMITED,N/A
5,AMAN HEALTH MAINTENANCE ORGANIZATIONS,https://amanhmo.com/home
6,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,https://anchorhmo.com/
7,ASHMED INTEGRATED HEALTH SERVICES LTD.,https://ashmedintegratedhealthservices.com/
8,AVON HEALTHCARE LIMITED,https://www.avonhealthcare.com/
9,AXA MANSARD HEALTH LIMITED,https://www.axamansard.com/health/plans/


In [61]:
print("Number of HMOs:", len(website_df))
print("Websites found:", (website_df["Website"] != "N/A").sum())

display(website_df.head(15))

Number of HMOs: 95
Websites found: 88


,Name,Website
0,HMO,N/A
1,A&M HEALTHCARE TRUST LIMITED,https://amhmo.com/
2,AIICO MULTISHIELD NIGERIA LIMITED,https://www.aiicomultishield.com/
3,ALLEANZA HEALTH MANAGEMENT LIMITED,https://alleanzahealth.com/individual-plans/
4,ALLY HEALTHCARE LIMITED,N/A
5,AMAN HEALTH MAINTENANCE ORGANIZATIONS,https://amanhmo.com/home
6,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,https://anchorhmo.com/
7,ASHMED INTEGRATED HEALTH SERVICES LTD.,https://ashmedintegratedhealthservices.com/
8,AVON HEALTHCARE LIMITED,https://www.avonhealthcare.com/
9,AXA MANSARD HEALTH LIMITED,https://www.axamansard.com/health/plans/


In [62]:
website_df = website_df[website_df["Name"] != "HMO"].reset_index(drop=True)

print("Number of HMOs:", len(website_df))
print("Websites found:", (website_df["Website"] != "N/A").sum())

display(website_df)

Number of HMOs: 94
Websites found: 88


,Name,Website
0,A&M HEALTHCARE TRUST LIMITED,https://amhmo.com/
1,AIICO MULTISHIELD NIGERIA LIMITED,https://www.aiicomultishield.com/
2,ALLEANZA HEALTH MANAGEMENT LIMITED,https://alleanzahealth.com/individual-plans/
3,ALLY HEALTHCARE LIMITED,N/A
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,https://amanhmo.com/home
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,https://anchorhmo.com/
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,https://ashmedintegratedhealthservices.com/
7,AVON HEALTHCARE LIMITED,https://www.avonhealthcare.com/
8,AXA MANSARD HEALTH LIMITED,https://www.axamansard.com/health/plans/
9,BASTION HEALTH LIMITED,https://bastionhmo.com/


In [63]:
# Check the number of HMOs
print("Number of HMOs:", len(nigeria_hmo))

# Check for duplicate names
print("Duplicate names:", nigeria_hmo["Name"].duplicated().sum())

# Show the full list
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

nigeria_hmo[["Name", "Website"]]

Number of HMOs: 94
Duplicate names: 0


,Name,Website
0,A&M HEALTHCARE TRUST LIMITED,https://amhmo.com/
1,AIICO MULTISHIELD NIGERIA LIMITED,https://www.aiicomultishield.com/
2,ALLEANZA HEALTH MANAGEMENT LIMITED,https://alleanzahealth.com/individual-plans/
3,ALLY HEALTHCARE LIMITED,https://amanhmo.com/home
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,https://anchorhmo.com/
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,https://ashmedintegratedhealthservices.com/
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,https://www.avonhealthcare.com/
7,AVON HEALTHCARE LIMITED,https://www.axamansard.com/health/plans/
8,AXA MANSARD HEALTH LIMITED,https://bastionhmo.com/
9,BASTION HEALTH LIMITED,https://www.centurymedicaid.org/


In [64]:
nigeria_hmo[nigeria_hmo["Website"].str.contains(
    "Infinitex2|princeton|fountainhealthcareng|farepharm|nnpcgroup|libertyhealth|sunugroup",
    case=False,
    na=False
)][["Name", "Website"]]

,Name,Website
15,DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO,https://fountainhealthcareng.com/office/
27,HEALTHSPRING HMO,"http://Infinitex2health@gmail.com,%20info@infinitex2health.net"
40,MASSLIFE HEALTHCARE LIMITED,https://www.farepharm.com/hmo/143
44,MEDIPLAN HEALTHCARE LIMITED,https://nnpcgroup.com/contact
53,PHILLIPS HEALTH MANAGEMENT SERVICES LIMITED,http://info@princetonhmo.net
58,PROHEALTH HMO LTD,https://fountainhealthcareng.com/office/
68,SKYDA HEALTH LIMITED,http://nigeria.health@sunugroup.com.
70,SPRINGTIDE HEALTHCARE SERVICES LTD.,https://www.libertyhealth.net/nigeria/en/


In [65]:
website_corrections = {
    "HEALTHSPRING HMO": "https://healthspringhmo.com/",
    "MASSLIFE HEALTHCARE LIMITED": "https://masslife.com.ng/",
    "MEDIPLAN HEALTHCARE LIMITED": "https://mediplanhealthcare.com/",
    "PROHEALTH HMO LTD": "https://www.prohealthhmo.com/"
}

for company, website in website_corrections.items():
    nigeria_hmo.loc[
        nigeria_hmo["Name"] == company, "Website"
    ] = website

nigeria_hmo[nigeria_hmo["Name"].isin(website_corrections.keys())][
    ["Name", "Website"]
]

,Name,Website
27,HEALTHSPRING HMO,https://healthspringhmo.com/
40,MASSLIFE HEALTHCARE LIMITED,https://masslife.com.ng/
44,MEDIPLAN HEALTHCARE LIMITED,https://mediplanhealthcare.com/
58,PROHEALTH HMO LTD,https://www.prohealthhmo.com/


In [66]:
website_corrections = {
    "HEALTHSPRING HMO": "https://healthspringhmo.com/",
    "MASSLIFE HEALTHCARE LIMITED": "https://masslife.com.ng/",
    "MEDIPLAN HEALTHCARE LIMITED": "https://mediplanhealthcare.com/",
    "PROHEALTH HMO LTD": "https://www.prohealthhmo.com/",
    "PHILLIPS HEALTH MANAGEMENT SERVICES LIMITED": "https://phillipshmo.net/",
    "SKYDA HEALTH LIMITED": "https://skydalink.com/",
    "SPRINGTIDE HEALTHCARE SERVICES LTD.": "https://springtidehmo.com/"
}

for company, website in website_corrections.items():
    nigeria_hmo.loc[
        nigeria_hmo["Name"] == company, "Website"
    ] = website
    
nigeria_hmo.loc[
    nigeria_hmo["Name"] == "DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO",
    "Website"
] = "N/A"

nigeria_hmo[nigeria_hmo["Name"].isin(
    list(website_corrections.keys()) +
    ["DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO"]
)][["Name", "Website"]]

,Name,Website
15,DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO,N/A
27,HEALTHSPRING HMO,https://healthspringhmo.com/
40,MASSLIFE HEALTHCARE LIMITED,https://masslife.com.ng/
44,MEDIPLAN HEALTHCARE LIMITED,https://mediplanhealthcare.com/
53,PHILLIPS HEALTH MANAGEMENT SERVICES LIMITED,https://phillipshmo.net/
58,PROHEALTH HMO LTD,https://www.prohealthhmo.com/
68,SKYDA HEALTH LIMITED,https://skydalink.com/
70,SPRINGTIDE HEALTHCARE SERVICES LTD.,https://springtidehmo.com/


In [67]:
nigeria_hmo.columns

Index(['S/NO.', 'Name', 'HMO ID', 'Website', 'Address', 'Email',
       'Contact Number', 'Country', 'Source', 'Source URL', 'State', 'Region',
       'Coverage State', 'Coverage Type', 'Services Rendered',
       'Target Customers'],
      dtype='object')

In [68]:
nigeria_hmo.head()

,S/NO.,Name,HMO ID,Website,Address,Email,Contact Number,Country,Source,Source URL,State,Region,Coverage State,Coverage Type,Services Rendered,Target Customers
0,1,A&M HEALTHCARE TRUST LIMITED,102,https://amhmo.com/,"Plot U Bekaji Road, Opposite Bekaji Jumma'at Mosque, Jimeta, Yola, Adamawa State",info@amhmo.com,"08033646497, 09162788582, 08024143666",Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,N/A,N/A,N/A,N/A,N/A,N/A
1,2,AIICO MULTISHIELD NIGERIA LIMITED,6,https://www.aiicomultishield.com/,"322, IKORODU ROAD, ANTHONY",info@aiicomultishield.com loshunniyi@aiicomultishield.com,O7026744353 O8056744353,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,N/A,N/A,N/A,N/A,N/A,N/A
2,3,ALLEANZA HEALTH MANAGEMENT LIMITED,111,https://alleanzahealth.com/individual-plans/,"? 83B, Basheer Shittu Avenue, Magodo Estate, Phase 2, Shangisha, Lagos",info@alleanzahealth.com,O7036592835,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,N/A,N/A,N/A,N/A,N/A,N/A
3,4,ALLY HEALTHCARE LIMITED,119,https://amanhmo.com/home,"Suit C4 Plot 1196 Ndjamena Crescent, Wuse II, Abuja",allyhealthcarelimited@gmail.com,O8148808171,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,N/A,N/A,N/A,N/A,N/A,N/A
4,5,AMAN HEALTH MAINTENANCE ORGANIZATIONS,121,https://anchorhmo.com/,"? 1, M. M. Alkali Street, Off 442 Crescent, Citec Vilas Gwarinpa, Abuja",info@amanhmo.com,O8100588906,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,N/A,N/A,N/A,N/A,N/A,N/A


In [69]:
nigeria_hmo["Contact Number"] = (
    nigeria_hmo["Contact Number"]
    .astype(str)
    .str.replace("O", "0", regex=False)
)

In [70]:
nigeria_hmo[["Name", "Email", "Contact Number"]].head(10)

,Name,Email,Contact Number
0,A&M HEALTHCARE TRUST LIMITED,info@amhmo.com,"08033646497, 09162788582, 08024143666"
1,AIICO MULTISHIELD NIGERIA LIMITED,info@aiicomultishield.com loshunniyi@aiicomultishield.com,07026744353 08056744353
2,ALLEANZA HEALTH MANAGEMENT LIMITED,info@alleanzahealth.com,07036592835
3,ALLY HEALTHCARE LIMITED,allyhealthcarelimited@gmail.com,08148808171
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,info@amanhmo.com,08100588906
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,info@anchorhmo.com,"08031230306, 07058890062, 07080601192"
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,info@ashmedintegratedhealthservices.com,07036035182
7,AVON HEALTHCARE LIMITED,info@avonhealthcare.com,"08102659972, 07002779800"
8,AXA MANSARD HEALTH LIMITED,healthcare@axamansard.com,0700AXAMANSARD
9,BASTION HEALTH LIMITED,Naomiduku@bastionhmo.com,0802BASTI0NHM0 08002278466


In [71]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

nigeria_hmo[["Name", "Address"]]

,Name,Address
0,A&M HEALTHCARE TRUST LIMITED,"Plot U Bekaji Road, Opposite Bekaji Jumma'at Mosque, Jimeta, Yola, Adamawa State"
1,AIICO MULTISHIELD NIGERIA LIMITED,"322, IKORODU ROAD, ANTHONY"
2,ALLEANZA HEALTH MANAGEMENT LIMITED,"? 83B, Basheer Shittu Avenue, Magodo Estate, Phase 2, Shangisha, Lagos"
3,ALLY HEALTHCARE LIMITED,"Suit C4 Plot 1196 Ndjamena Crescent, Wuse II, Abuja"
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,"? 1, M. M. Alkali Street, Off 442 Crescent, Citec Vilas Gwarinpa, Abuja"
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,GOLDCREST MALL BUILDING
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,"RE 013 ZUNGERU CLOSE BY BIMA ROAD NITEL QUARTERS, KADUNA"
7,AVON HEALTHCARE LIMITED,"22B, GLOVER ROAD"
8,AXA MANSARD HEALTH LIMITED,"177, IKORODU ROAD, ONIPANU, LAGOS"
9,BASTION HEALTH LIMITED,"UBN HOUSE NO 1 ADEOLA ODEKU STREET, VICTORIA ISLAND, LAGOS, ETI OSA, LAGOS, Nigeria"


In [72]:
state_mapping = {
    "A&M HEALTHCARE TRUST LIMITED": "Adamawa",
    "AIICO MULTISHIELD NIGERIA LIMITED": "Lagos",
    "ALLEANZA HEALTH MANAGEMENT LIMITED": "Lagos",
    "ALLY HEALTHCARE LIMITED": "FCT",
    "AMAN HEALTH MAINTENANCE ORGANIZATIONS": "FCT",
    "ANCHOR HMO INTERNATIONAL COMPANY LIMITED": "N/A",
    "ASHMED INTEGRATED HEALTH SERVICES LTD.": "Kaduna",
    "AVON HEALTHCARE LIMITED": "Lagos",
    "AXA MANSARD HEALTH LIMITED": "Lagos",
    "BASTION HEALTH LIMITED": "Lagos",
    "BONITAS HEALTH MAINTENANCE LIMITED": "Rivers",
    "CENTURY MEDICAID SERVICES LIMITED": "Rivers",
    "CLEARLINE INTERNATIONAL LIMITED": "Lagos",
    "DEFENCE HEALTH MAINTENANCE LIMITED": "FCT",
    "Delog Medical Services Ltd. (HMO)": "Lagos",
    "DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO": "Lagos",
    "DOT HMO Ltd.": "Lagos",
    "FOUNTAIN HEALTHCARE LIMITED": "Lagos",
    "GNI HEALTHCARE LTD": "Lagos",
    "GORAH HEALTHCARE LTD.": "Lagos",
    "GREENBAY HEALTHCARE SERVICES LIMITED": "Lagos",
    "GREENFIELD HEALTH MANAGEMENT LTD": "N/A",
    "GROOMING HEALTH MANAGEMENT LIMITED": "Lagos",
    "HALLMARK HEALTH SERVICES LIMITED": "Lagos",
    "HEALTH ASSUR LIMITED": "Lagos",
    "Health Partners Limited": "N/A",
    "HEALTHCARE INTERNATIONAL LIMITED": "Lagos",
    "HEALTHSPRING HMO": "FCT",
    "HYGEIA HMO LIMITED": "Lagos",
    "Infinite X2 Health Maintenance Services": "Kaduna",
    "INTEGRATED HEALTHCARE": "FCT",
    "INTERNATIONAL HEALTH MGT. SERVICES": "FCT",
    "IVES MEDICARE": "Lagos",
    "KENNEDIA HMO LIMITED": "Lagos",
    "LEADWAY HEALTH LIMITED": "Lagos",
    "LIFE WORTH MEDICARE LTD": "Lagos",
    "LIFESAVER HEALTHCARE LIMITED": "Lagos",
    "MAAYOIT HEALTH CARE LIMITED": "Kwara",
    "MARINA MEDICAL SERVICES HMO LIMITED": "Lagos",
    "MARKFEMA NIGERIA LTD.": "FCT",
    "MASSLIFE HEALTHCARE LIMITED": "FCT",
    "MB & O HEALTHCARE SERVICES LIMITED": "Lagos",
    "MEDEXIA LIMITED": "Lagos",
    "MEDICARE ALLIANCE LIMITED": "FCT",
    "MEDIPLAN HEALTHCARE LIMITED": "Lagos",
    "METROHEALTH HMO LIMITED": "Lagos",
    "NEM HEALTH LIMITED": "Lagos",
    "NNPC-HMO LIMITED": "N/A",
    "NONSUCH MEDICARE LIMITED": "Oyo",
    "NOOR HEALTH LTD.": "Lagos",
    "NOVO HEALTH AFRICA LIMITED": "Lagos",
    "OCEANIC HEALTH MANAGEMENT LIMITED": "Lagos / FCT",
    "PERAMARE HEALTH MANAGEMENT COMPANY LIMITED": "FCT",
    "PHILLIPS HEALTH MANAGEMENT SERVICES LIMITED": "Lagos",
    "POLICE HEALTH MAINTENANCE LIMITED": "FCT",
    "PRECIOUS HEALTHCARE LIMITED": "FCT",
    "PREPAID MEDICARE SERVICES LTD.": "FCT",
    "PRINCETON HEALTH": "Oyo",
    "PROHEALTH HMO LTD": "FCT",
    "REDCARE HEALTH SERVICES LIMITED": "N/A",
    "REGENIX HEALTH CARE SERVICE LIMITED": "Rivers",
    "RELIANCE HMO LIMITED": "Lagos",
    "RODING HEALTHCARE LTD.": "Lagos",
    "RONSBERGER NIGERIA LTD.": "FCT",
    "ROTHAUGE HEALTHCARE LIMITED": "Lagos",
    "ROYAL HEALTH MAINTENANCE SERVICES LTD.": "Imo",
    "SALUS TRUST GTE": "Lagos",
    "SERAPH HMO LIMITED": "Benue",
    "SKYDA HEALTH LIMITED": "FCT",
    "SONGHAI HEALTH TRUST": "FCT",
    "SPRINGTIDE HEALTHCARE SERVICES LTD.": "Rivers",
    "STERLING HEALTH MANAGED CARE SERVICES LIMITED": "Lagos",
    "SUNU HEALTH NIGERIA LIMITED": "Lagos",
    "SYNERGY WELLCARE MEDICAID LIMITED": "Rivers",
    "TOTAL HEALTH TRUST LIMITED": "Lagos",
    "ULTIMATE HEALTH MANAGEMENT SERVICES LTD": "FCT",
    "UNITED COMPREHENSIVE HEALTH MANAGERS LTD.": "Rivers",
    "UNITED HEALTHCARE INTERNATIONAL LIMITED": "FCT",
    "VENUS MEDICARE LTD": "FCT",
    "VERITAS HEALTHCARE LIMITED": "FCT",
    "WELL HEALTH NETWORK LIMITED": "Enugu",
    "WELLNESS HEALTH MANAGEMENT SERVICES LIMITED": "Lagos",
    "ZUMA HEALTH TRUST": "FCT",
    "Healthnomics HMO PLC": "FCT",
    "HYSSOP Health International Limited (HMO)": "Delta",
    "Hopewell Healthcare Management Ltd": "FCT",
    "Crown Jewel HMO Ltd.": "FCT",
    "Smathealth Medicare Limited": "Lagos",
    "Health Assur Limited": "Lagos",
    "Quest Medicare Limited": "Lagos",
    "Life Action Plus Ltd.": "Lagos",
    "Zenor Healthcare Ltd.": "Lagos",
    "Aspire HMO Limited": "Lagos",
    "Mitera Health Limited": "Lagos"
}

nigeria_hmo["State"] = nigeria_hmo["Name"].map(state_mapping)

nigeria_hmo[["Name", "Address", "State"]]

,Name,Address,State
0,A&M HEALTHCARE TRUST LIMITED,"Plot U Bekaji Road, Opposite Bekaji Jumma'at Mosque, Jimeta, Yola, Adamawa State",Adamawa
1,AIICO MULTISHIELD NIGERIA LIMITED,"322, IKORODU ROAD, ANTHONY",Lagos
2,ALLEANZA HEALTH MANAGEMENT LIMITED,"? 83B, Basheer Shittu Avenue, Magodo Estate, Phase 2, Shangisha, Lagos",Lagos
3,ALLY HEALTHCARE LIMITED,"Suit C4 Plot 1196 Ndjamena Crescent, Wuse II, Abuja",FCT
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,"? 1, M. M. Alkali Street, Off 442 Crescent, Citec Vilas Gwarinpa, Abuja",FCT
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,GOLDCREST MALL BUILDING,N/A
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,"RE 013 ZUNGERU CLOSE BY BIMA ROAD NITEL QUARTERS, KADUNA",Kaduna
7,AVON HEALTHCARE LIMITED,"22B, GLOVER ROAD",Lagos
8,AXA MANSARD HEALTH LIMITED,"177, IKORODU ROAD, ONIPANU, LAGOS",Lagos
9,BASTION HEALTH LIMITED,"UBN HOUSE NO 1 ADEOLA ODEKU STREET, VICTORIA ISLAND, LAGOS, ETI OSA, LAGOS, Nigeria",Lagos


In [73]:
region_map = {
    # North Central
    "Benue": "North Central",
    "Kogi": "North Central",
    "Kwara": "North Central",
    "Nasarawa": "North Central",
    "Niger": "North Central",
    "Plateau": "North Central",
    "FCT": "North Central",

    # North East
    "Adamawa": "North East",
    "Bauchi": "North East",
    "Borno": "North East",
    "Gombe": "North East",
    "Taraba": "North East",
    "Yobe": "North East",

    # North West
    "Jigawa": "North West",
    "Kaduna": "North West",
    "Kano": "North West",
    "Katsina": "North West",
    "Kebbi": "North West",
    "Sokoto": "North West",
    "Zamfara": "North West",

    # South West
    "Ekiti": "South West",
    "Lagos": "South West",
    "Ogun": "South West",
    "Ondo": "South West",
    "Osun": "South West",
    "Oyo": "South West",

    # South East
    "Abia": "South East",
    "Anambra": "South East",
    "Ebonyi": "South East",
    "Enugu": "South East",
    "Imo": "South East",

    # South South
    "Akwa Ibom": "South South",
    "Bayelsa": "South South",
    "Cross River": "South South",
    "Delta": "South South",
    "Edo": "South South",
    "Rivers": "South South"
}

nigeria_hmo["Region"] = nigeria_hmo["State"].map(region_map)

nigeria_hmo[["Name", "State", "Region"]]

,Name,State,Region
0,A&M HEALTHCARE TRUST LIMITED,Adamawa,North East
1,AIICO MULTISHIELD NIGERIA LIMITED,Lagos,South West
2,ALLEANZA HEALTH MANAGEMENT LIMITED,Lagos,South West
3,ALLY HEALTHCARE LIMITED,FCT,North Central
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,FCT,North Central
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,N/A,NaN
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Kaduna,North West
7,AVON HEALTHCARE LIMITED,Lagos,South West
8,AXA MANSARD HEALTH LIMITED,Lagos,South West
9,BASTION HEALTH LIMITED,Lagos,South West


In [74]:
nigeria_hmo["Region"] = nigeria_hmo["Region"].fillna("N/A")

In [75]:
nigeria_hmo[["Name", "State", "Region"]]

,Name,State,Region
0,A&M HEALTHCARE TRUST LIMITED,Adamawa,North East
1,AIICO MULTISHIELD NIGERIA LIMITED,Lagos,South West
2,ALLEANZA HEALTH MANAGEMENT LIMITED,Lagos,South West
3,ALLY HEALTHCARE LIMITED,FCT,North Central
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,FCT,North Central
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,N/A,N/A
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Kaduna,North West
7,AVON HEALTHCARE LIMITED,Lagos,South West
8,AXA MANSARD HEALTH LIMITED,Lagos,South West
9,BASTION HEALTH LIMITED,Lagos,South West


In [76]:
nigeria_hmo["Coverage State"] = "N/A"

In [77]:
nationwide_hmos = [
    "HEALTHSPRING HMO",
    "HYGEIA HMO LIMITED",
    "INTERNATIONAL HEALTH MGT. SERVICES"
]

nigeria_hmo.loc[
    nigeria_hmo["Name"].isin(nationwide_hmos),
    "Coverage State"
] = "Nationwide – Nigeria"

In [78]:
nigeria_hmo[["Name", "State", "Region", "Coverage State"]]

,Name,State,Region,Coverage State
0,A&M HEALTHCARE TRUST LIMITED,Adamawa,North East,N/A
1,AIICO MULTISHIELD NIGERIA LIMITED,Lagos,South West,N/A
2,ALLEANZA HEALTH MANAGEMENT LIMITED,Lagos,South West,N/A
3,ALLY HEALTHCARE LIMITED,FCT,North Central,N/A
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,FCT,North Central,N/A
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,N/A,N/A,N/A
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Kaduna,North West,N/A
7,AVON HEALTHCARE LIMITED,Lagos,South West,N/A
8,AXA MANSARD HEALTH LIMITED,Lagos,South West,N/A
9,BASTION HEALTH LIMITED,Lagos,South West,N/A


In [79]:
nigeria_hmo["Coverage State"] = "N/A"

In [80]:
coverage_state = {
    "A&M HEALTHCARE TRUST LIMITED": "N/A",
    "AIICO MULTISHIELD NIGERIA LIMITED": "Nationwide – Nigeria",
    "ALLEANZA HEALTH MANAGEMENT LIMITED": "Nationwide – Nigeria",
    "ALLY HEALTHCARE LIMITED": "N/A",
    "AMAN HEALTH MAINTENANCE ORGANIZATIONS": "N/A",
    "ANCHOR HMO INTERNATIONAL COMPANY LIMITED": "Nationwide – Nigeria",
    "ASHMED INTEGRATED HEALTH SERVICES LTD.": "N/A",
    "AVON HEALTHCARE LIMITED": "Nationwide – Nigeria",
    "AXA MANSARD HEALTH LIMITED": "Nationwide – Nigeria",
    "BASTION HEALTH LIMITED": "Nationwide – Nigeria",
    "BONITAS HEALTH MAINTENANCE LIMITED": "N/A",
    "CENTURY MEDICAID SERVICES LIMITED": "Nationwide – Nigeria",
    "CLEARLINE INTERNATIONAL LIMITED": "N/A",
    "DEFENCE HEALTH MAINTENANCE LIMITED": "N/A",
    "Delog Medical Services Ltd. (HMO)": "N/A",
    "DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO": "N/A",
    "DOT HMO Ltd.": "N/A",
    "FOUNTAIN HEALTHCARE LIMITED": "N/A",
    "GNI HEALTHCARE LTD": "N/A",
    "GORAH HEALTHCARE LTD.": "N/A",
    "GREENBAY HEALTHCARE SERVICES LIMITED": "N/A",
    "GREENFIELD HEALTH MANAGEMENT LTD": "N/A",
    "GROOMING HEALTH MANAGEMENT LIMITED": "Nationwide – Nigeria",
    "HALLMARK HEALTH SERVICES LIMITED": "N/A",
    "HEALTH ASSUR LIMITED": "Nationwide – Nigeria",
    "Health Partners Limited": "N/A",
    "HEALTHCARE INTERNATIONAL LIMITED": "N/A",
    "HEALTHSPRING HMO": "Nationwide – Nigeria",
    "HYGEIA HMO LIMITED": "Nationwide – Nigeria",
    "Infinite X2 Health Maintenance Services": "N/A",
    "INTEGRATED HEALTHCARE": "N/A",
    "INTERNATIONAL HEALTH MGT. SERVICES": "Nationwide – Nigeria",
    "IVES MEDICARE": "N/A",
    "KENNEDIA HMO LIMITED": "N/A",
    "LEADWAY HEALTH LIMITED": "Nationwide – Nigeria",
    "LIFE WORTH MEDICARE LTD": "N/A",
    "LIFESAVER HEALTHCARE LIMITED": "N/A",
    "MAAYOIT HEALTH CARE LIMITED": "N/A",
    "MARINA MEDICAL SERVICES HMO LIMITED": "N/A",
    "MARKFEMA NIGERIA LTD.": "N/A",
    "MASSLIFE HEALTHCARE LIMITED": "N/A",
    "MB & O HEALTHCARE SERVICES LIMITED": "N/A",
    "MEDEXIA LIMITED": "N/A",
    "MEDICARE ALLIANCE LIMITED": "N/A",
    "MEDIPLAN HEALTHCARE LIMITED": "Nationwide – Nigeria",
    "METROHEALTH HMO LIMITED": "N/A",
    "NEM HEALTH LIMITED": "Nationwide – Nigeria",
    "NNPC-HMO LIMITED": "N/A",
    "NONSUCH MEDICARE LIMITED": "N/A",
    "NOOR HEALTH LTD.": "N/A",
    "NOVO HEALTH AFRICA LIMITED": "N/A",
    "OCEANIC HEALTH MANAGEMENT LIMITED": "N/A",
    "PERAMARE HEALTH MANAGEMENT COMPANY LIMITED": "N/A",
    "PHILLIPS HEALTH MANAGEMENT SERVICES LIMITED": "N/A",
    "POLICE HEALTH MAINTENANCE LIMITED": "N/A",
    "PRECIOUS HEALTHCARE LIMITED": "N/A",
    "PREPAID MEDICARE SERVICES LTD.": "N/A",
    "PRINCETON HEALTH": "N/A",
    "PROHEALTH HMO LTD": "N/A",
    "REDCARE HEALTH SERVICES LIMITED": "N/A",
    "REGENIX HEALTH CARE SERVICE LIMITED": "N/A",
    "RELIANCE HMO LIMITED": "N/A",
    "RODING HEALTHCARE LTD.": "N/A",
    "RONSBERGER NIGERIA LTD.": "N/A",
    "ROTHAUGE HEALTHCARE LIMITED": "Nationwide – Nigeria",
    "ROYAL HEALTH MAINTENANCE SERVICES LTD.": "N/A",
    "SALUS TRUST GTE": "Nationwide – Nigeria",
    "SERAPH HMO LIMITED": "N/A",
    "SKYDA HEALTH LIMITED": "N/A",
    "SONGHAI HEALTH TRUST": "Nationwide – Nigeria",
    "SPRINGTIDE HEALTHCARE SERVICES LTD.": "Nationwide – Nigeria",
    "STERLING HEALTH MANAGED CARE SERVICES LIMITED": "N/A",
    "SUNU HEALTH NIGERIA LIMITED": "Nationwide – Nigeria",
    "SYNERGY WELLCARE MEDICAID LIMITED": "Nationwide – Nigeria",
    "TOTAL HEALTH TRUST LIMITED": "N/A",
    "ULTIMATE HEALTH MANAGEMENT SERVICES LTD": "Nationwide – Nigeria",
    "UNITED COMPREHENSIVE HEALTH MANAGERS LTD.": "N/A",
    "UNITED HEALTHCARE INTERNATIONAL LIMITED": "Nationwide – Nigeria",
    "VENUS MEDICARE LTD": "Nationwide – Nigeria",
    "VERITAS HEALTHCARE LIMITED": "Nationwide – Nigeria",
    "WELL HEALTH NETWORK LIMITED": "N/A",
    "WELLNESS HEALTH MANAGEMENT SERVICES LIMITED": "N/A",
    "ZUMA HEALTH TRUST": "N/A",
    "Healthnomics HMO PLC": "N/A",
    "HYSSOP Health International Limited (HMO)": "N/A",
    "Hopewell Healthcare Management Ltd": "N/A",
    "Crown Jewel HMO Ltd.": "N/A",
    "Smathealth Medicare Limited": "N/A",
    "Health Assur Limited": "Nationwide – Nigeria",
    "Quest Medicare Limited": "N/A",
    "Life Action Plus Ltd.": "N/A",
    "Zenor Healthcare Ltd.": "N/A",
    "Aspire HMO Limited": "N/A",
    "Mitera Health Limited": "N/A"
}

nigeria_hmo["Coverage State"] = (
    nigeria_hmo["Name"].map(coverage_state).fillna("N/A")
)

In [81]:
nigeria_hmo[["Name", "State", "Region", "Coverage State"]]

,Name,State,Region,Coverage State
0,A&M HEALTHCARE TRUST LIMITED,Adamawa,North East,N/A
1,AIICO MULTISHIELD NIGERIA LIMITED,Lagos,South West,Nationwide – Nigeria
2,ALLEANZA HEALTH MANAGEMENT LIMITED,Lagos,South West,Nationwide – Nigeria
3,ALLY HEALTHCARE LIMITED,FCT,North Central,N/A
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,FCT,North Central,N/A
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,N/A,N/A,Nationwide – Nigeria
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Kaduna,North West,N/A
7,AVON HEALTHCARE LIMITED,Lagos,South West,Nationwide – Nigeria
8,AXA MANSARD HEALTH LIMITED,Lagos,South West,Nationwide – Nigeria
9,BASTION HEALTH LIMITED,Lagos,South West,Nationwide – Nigeria


In [82]:
coverage_state = {
    "A&M HEALTHCARE TRUST LIMITED": "N/A",
    "AIICO MULTISHIELD NIGERIA LIMITED": "Nationwide – Nigeria",
    "ALLEANZA HEALTH MANAGEMENT LIMITED": "Nationwide – Nigeria",
    "ALLY HEALTHCARE LIMITED": "N/A",
    "AMAN HEALTH MAINTENANCE ORGANIZATIONS": "N/A",
    "ANCHOR HMO INTERNATIONAL COMPANY LIMITED": "Nationwide – Nigeria",
    "ASHMED INTEGRATED HEALTH SERVICES LTD.": "N/A",
    "AVON HEALTHCARE LIMITED": "Nationwide – Nigeria",
    "AXA MANSARD HEALTH LIMITED": "Nationwide – Nigeria",
    "BASTION HEALTH LIMITED": "Nationwide – Nigeria",
    "BONITAS HEALTH MAINTENANCE LIMITED": "N/A",
    "CENTURY MEDICAID SERVICES LIMITED": "Nationwide – Nigeria",
    "CLEARLINE INTERNATIONAL LIMITED": "N/A",
    "DEFENCE HEALTH MAINTENANCE LIMITED": "All 36 States + FCT",
    "Delog Medical Services Ltd. (HMO)": "N/A",
    "DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO": "N/A",
    "DOT HMO Ltd.": "N/A",
    "FOUNTAIN HEALTHCARE LIMITED": "N/A",
    "GNI HEALTHCARE LTD": "N/A",
    "GORAH HEALTHCARE LTD.": "Nationwide – Nigeria",
    "GREENBAY HEALTHCARE SERVICES LIMITED": "N/A",
    "GREENFIELD HEALTH MANAGEMENT LTD": "N/A",
    "GROOMING HEALTH MANAGEMENT LIMITED": "N/A",
    "HALLMARK HEALTH SERVICES LIMITED": "N/A",
    "HEALTH ASSUR LIMITED": "Nationwide – Nigeria",
    "Health Partners Limited": "N/A",
    "HEALTHCARE INTERNATIONAL LIMITED": "N/A",
    "HEALTHSPRING HMO": "Nationwide – Nigeria",
    "HYGEIA HMO LIMITED": "Nationwide – Nigeria",
    "Infinite X2 Health Maintenance Services": "N/A",
    "INTEGRATED HEALTHCARE": "N/A",
    "INTERNATIONAL HEALTH MGT. SERVICES": "Nationwide – Nigeria",
    "IVES MEDICARE": "N/A",
    "KENNEDIA HMO LIMITED": "N/A",
    "LEADWAY HEALTH LIMITED": "Nationwide – Nigeria",
    "LIFE WORTH MEDICARE LTD": "N/A",
    "LIFESAVER HEALTHCARE LIMITED": "N/A",
    "MAAYOIT HEALTH CARE LIMITED": "N/A",
    "MARINA MEDICAL SERVICES HMO LIMITED": "N/A",
    "MARKFEMA NIGERIA LTD.": "N/A",
    "MASSLIFE HEALTHCARE LIMITED": "N/A",
    "MB & O HEALTHCARE SERVICES LIMITED": "N/A",
    "MEDEXIA LIMITED": "N/A",
    "MEDICARE ALLIANCE LIMITED": "N/A",
    "MEDIPLAN HEALTHCARE LIMITED": "Nationwide – Nigeria",
    "METROHEALTH HMO LIMITED": "N/A",
    "NEM HEALTH LIMITED": "Nationwide – Nigeria",
    "NNPC-HMO LIMITED": "N/A",
    "NONSUCH MEDICARE LIMITED": "N/A",
    "NOOR HEALTH LTD.": "N/A",
    "NOVO HEALTH AFRICA LIMITED": "N/A",
    "OCEANIC HEALTH MANAGEMENT LIMITED": "N/A",
    "PERAMARE HEALTH MANAGEMENT COMPANY LIMITED": "N/A",
    "PHILLIPS HEALTH MANAGEMENT SERVICES LIMITED": "N/A",
    "POLICE HEALTH MAINTENANCE LIMITED": "N/A",
    "PRECIOUS HEALTHCARE LIMITED": "N/A",
    "PREPAID MEDICARE SERVICES LTD.": "N/A",
    "PRINCETON HEALTH": "N/A",
    "PROHEALTH HMO LTD": "N/A",
    "REDCARE HEALTH SERVICES LIMITED": "N/A",
    "REGENIX HEALTH CARE SERVICE LIMITED": "N/A",
    "RELIANCE HMO LIMITED": "N/A",
    "RODING HEALTHCARE LTD.": "N/A",
    "RONSBERGER NIGERIA LTD.": "N/A",
    "ROTHAUGE HEALTHCARE LIMITED": "Nationwide – Nigeria",
    "ROYAL HEALTH MAINTENANCE SERVICES LTD.": "N/A",
    "SALUS TRUST GTE": "Nationwide – Nigeria",
    "SERAPH HMO LIMITED": "N/A",
    "SKYDA HEALTH LIMITED": "N/A",
    "SONGHAI HEALTH TRUST": "Nationwide – Nigeria",
    "SPRINGTIDE HEALTHCARE SERVICES LTD.": "Nationwide – Nigeria",
    "STERLING HEALTH MANAGED CARE SERVICES LIMITED": "N/A",
    "SUNU HEALTH NIGERIA LIMITED": "Nationwide – Nigeria",
    "SYNERGY WELLCARE MEDICAID LIMITED": "Nationwide – Nigeria",
    "TOTAL HEALTH TRUST LIMITED": "N/A",
    "ULTIMATE HEALTH MANAGEMENT SERVICES LTD": "Nationwide – Nigeria",
    "UNITED COMPREHENSIVE HEALTH MANAGERS LTD.": "N/A",
    "UNITED HEALTHCARE INTERNATIONAL LIMITED": "Nationwide – Nigeria",
    "VENUS MEDICARE LTD": "Nationwide – Nigeria",
    "VERITAS HEALTHCARE LIMITED": "Nationwide – Nigeria",
    "WELL HEALTH NETWORK LIMITED": "N/A",
    "WELLNESS HEALTH MANAGEMENT SERVICES LIMITED": "N/A",
    "ZUMA HEALTH TRUST": "N/A",
    "Healthnomics HMO PLC": "N/A",
    "HYSSOP Health International Limited (HMO)": "N/A",
    "Hopewell Healthcare Management Ltd": "N/A",
    "Crown Jewel HMO Ltd.": "N/A",
    "Smathealth Medicare Limited": "N/A",
    "Health Assur Limited": "Nationwide – Nigeria",
    "Quest Medicare Limited": "N/A",
    "Life Action Plus Ltd.": "N/A",
    "Zenor Healthcare Ltd.": "N/A",
    "Aspire HMO Limited": "N/A",
    "Mitera Health Limited": "N/A"
}

nigeria_hmo["Coverage State"] = (
    nigeria_hmo["Name"].map(coverage_state).fillna("N/A")
)

In [83]:
nigeria_hmo[["Name", "State", "Coverage State"]]

,Name,State,Coverage State
0,A&M HEALTHCARE TRUST LIMITED,Adamawa,N/A
1,AIICO MULTISHIELD NIGERIA LIMITED,Lagos,Nationwide – Nigeria
2,ALLEANZA HEALTH MANAGEMENT LIMITED,Lagos,Nationwide – Nigeria
3,ALLY HEALTHCARE LIMITED,FCT,N/A
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,FCT,N/A
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,N/A,Nationwide – Nigeria
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Kaduna,N/A
7,AVON HEALTHCARE LIMITED,Lagos,Nationwide – Nigeria
8,AXA MANSARD HEALTH LIMITED,Lagos,Nationwide – Nigeria
9,BASTION HEALTH LIMITED,Lagos,Nationwide – Nigeria


In [84]:
nigeria_hmo["Coverage State"].value_counts()

Coverage State
N/A                     67
Nationwide – Nigeria    26
All 36 States + FCT      1
Name: count, dtype: int64

In [85]:
coverage_na = nigeria_hmo[nigeria_hmo["Coverage State"] == "N/A"][["Name", "State", "Website"]]

coverage_na

,Name,State,Website
0,A&M HEALTHCARE TRUST LIMITED,Adamawa,https://amhmo.com/
3,ALLY HEALTHCARE LIMITED,FCT,https://amanhmo.com/home
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,FCT,https://anchorhmo.com/
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Kaduna,https://www.avonhealthcare.com/
10,BONITAS HEALTH MAINTENANCE LIMITED,Rivers,https://clearlinehmo.com/
12,CLEARLINE INTERNATIONAL LIMITED,Lagos,https://delogmedicalhmo.com/
14,Delog Medical Services Ltd. (HMO),Lagos,https://hmo.dot.ai/
15,DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO,Lagos,N/A
16,DOT HMO Ltd.,Lagos,http://gnihealthcare.com/
17,FOUNTAIN HEALTHCARE LIMITED,Lagos,https://gorahhmo.com/


In [86]:
print("Number of N/A:", len(coverage_na))

Number of N/A: 67


In [87]:
coverage_na.to_string(index=False)

'                                         Name       State                                Website\n                 A&M HEALTHCARE TRUST LIMITED     Adamawa                     https://amhmo.com/\n                      ALLY HEALTHCARE LIMITED         FCT               https://amanhmo.com/home\n        AMAN HEALTH MAINTENANCE ORGANIZATIONS         FCT                 https://anchorhmo.com/\n       ASHMED INTEGRATED HEALTH SERVICES LTD.      Kaduna        https://www.avonhealthcare.com/\n           BONITAS HEALTH MAINTENANCE LIMITED      Rivers              https://clearlinehmo.com/\n              CLEARLINE INTERNATIONAL LIMITED       Lagos           https://delogmedicalhmo.com/\n            Delog Medical Services Ltd. (HMO)       Lagos                    https://hmo.dot.ai/\n  DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO       Lagos                                    N/A\n                                 DOT HMO Ltd.       Lagos              http://gnihealthcare.com/\n                  F

In [88]:
coverage_na = nigeria_hmo[
    nigeria_hmo["Coverage State"] == "N/A"
][["Name", "State", "Website"]].copy()

coverage_na

,Name,State,Website
0,A&M HEALTHCARE TRUST LIMITED,Adamawa,https://amhmo.com/
3,ALLY HEALTHCARE LIMITED,FCT,https://amanhmo.com/home
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,FCT,https://anchorhmo.com/
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Kaduna,https://www.avonhealthcare.com/
10,BONITAS HEALTH MAINTENANCE LIMITED,Rivers,https://clearlinehmo.com/
12,CLEARLINE INTERNATIONAL LIMITED,Lagos,https://delogmedicalhmo.com/
14,Delog Medical Services Ltd. (HMO),Lagos,https://hmo.dot.ai/
15,DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO,Lagos,N/A
16,DOT HMO Ltd.,Lagos,http://gnihealthcare.com/
17,FOUNTAIN HEALTHCARE LIMITED,Lagos,https://gorahhmo.com/


In [89]:
nigeria_hmo[["Name", "Website"]].to_string(index=False)

'                                         Name                                      Website\n                 A&M HEALTHCARE TRUST LIMITED                           https://amhmo.com/\n            AIICO MULTISHIELD NIGERIA LIMITED            https://www.aiicomultishield.com/\n           ALLEANZA HEALTH MANAGEMENT LIMITED https://alleanzahealth.com/individual-plans/\n                      ALLY HEALTHCARE LIMITED                     https://amanhmo.com/home\n        AMAN HEALTH MAINTENANCE ORGANIZATIONS                       https://anchorhmo.com/\n     ANCHOR HMO INTERNATIONAL COMPANY LIMITED  https://ashmedintegratedhealthservices.com/\n       ASHMED INTEGRATED HEALTH SERVICES LTD.              https://www.avonhealthcare.com/\n                      AVON HEALTHCARE LIMITED     https://www.axamansard.com/health/plans/\n                   AXA MANSARD HEALTH LIMITED                      https://bastionhmo.com/\n                       BASTION HEALTH LIMITED             https://www.centuryme

In [90]:
nigeria_hmo[["S/NO.", "Name", "Website"]].head(20)

,S/NO.,Name,Website
0,1,A&M HEALTHCARE TRUST LIMITED,https://amhmo.com/
1,2,AIICO MULTISHIELD NIGERIA LIMITED,https://www.aiicomultishield.com/
2,3,ALLEANZA HEALTH MANAGEMENT LIMITED,https://alleanzahealth.com/individual-plans/
3,4,ALLY HEALTHCARE LIMITED,https://amanhmo.com/home
4,5,AMAN HEALTH MAINTENANCE ORGANIZATIONS,https://anchorhmo.com/
5,6,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,https://ashmedintegratedhealthservices.com/
6,7,ASHMED INTEGRATED HEALTH SERVICES LTD.,https://www.avonhealthcare.com/
7,8,AVON HEALTHCARE LIMITED,https://www.axamansard.com/health/plans/
8,9,AXA MANSARD HEALTH LIMITED,https://bastionhmo.com/
9,10,BASTION HEALTH LIMITED,https://www.centurymedicaid.org/


In [93]:
nigeria_hmo[['Name', 'State', 'Region', 'Coverage State']]

,Name,State,Region,Coverage State
0,A&M HEALTHCARE TRUST LIMITED,Adamawa,North East,N/A
1,AIICO MULTISHIELD NIGERIA LIMITED,Lagos,South West,Nationwide – Nigeria
2,ALLEANZA HEALTH MANAGEMENT LIMITED,Lagos,South West,Nationwide – Nigeria
3,ALLY HEALTHCARE LIMITED,FCT,North Central,N/A
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,FCT,North Central,N/A
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,N/A,N/A,Nationwide – Nigeria
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Kaduna,North West,N/A
7,AVON HEALTHCARE LIMITED,Lagos,South West,Nationwide – Nigeria
8,AXA MANSARD HEALTH LIMITED,Lagos,South West,Nationwide – Nigeria
9,BASTION HEALTH LIMITED,Lagos,South West,Nationwide – Nigeria


In [95]:
nigeria_hmo['Coverage State'].value_counts(dropna=False)

Coverage State
N/A                     67
Nationwide – Nigeria    26
All 36 States + FCT      1
Name: count, dtype: int64

In [98]:
# ==========================================
# FINAL COVERAGE STATE CLEANUP
# ==========================================

coverage_map = {
    'A&M HEALTHCARE TRUST LIMITED': 'Adamawa',
    'AIICO MULTISHIELD NIGERIA LIMITED': 'Nationwide – Nigeria',
    'ALLEANZA HEALTH MANAGEMENT LIMITED': 'Nationwide – Nigeria',
    'ALLY HEALTHCARE LIMITED': 'FCT',
    'AMAN HEALTH MAINTENANCE ORGANIZATIONS': 'FCT',
    'ANCHOR HMO INTERNATIONAL COMPANY LIMITED': 'Nationwide – Nigeria',
    'ASHMED INTEGRATED HEALTH SERVICES LTD.': 'Kaduna',
    'AVON HEALTHCARE LIMITED': 'Nationwide – Nigeria',
    'AXA MANSARD HEALTH LIMITED': 'Nationwide – Nigeria',
    'BASTION HEALTH LIMITED': 'Lagos',
    'BONITAS HEALTH MAINTENANCE LIMITED': 'Rivers',
    'CENTURY MEDICAID SERVICES LIMITED': 'Rivers',
    'CLEARLINE INTERNATIONAL LIMITED': 'Nationwide – Nigeria',
    'DEFENCE HEALTH MAINTENANCE LIMITED': 'Nationwide – Nigeria',
    'Delog Medical Services Ltd. (HMO)': 'Lagos',
    'DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO': 'Lagos',
    'DOT HMO Ltd.': 'Nationwide – Nigeria',
    'FOUNTAIN HEALTHCARE LIMITED': 'Nationwide – Nigeria',
    'GNI HEALTHCARE LTD': 'Nationwide – Nigeria',
    'GORAH HEALTHCARE LTD.': 'Lagos',
    'GREENBAY HEALTHCARE SERVICES LIMITED': 'Nationwide – Nigeria',
    'GREENFIELD HEALTH MANAGEMENT LTD': 'Nationwide – Nigeria',
    'GROOMING HEALTH MANAGEMENT LIMITED': 'Lagos',
    'HALLMARK HEALTH SERVICES LIMITED': 'Lagos',
    'HEALTH ASSUR LIMITED': 'Nationwide – Nigeria',
    'Health Partners Limited': 'Nationwide – Nigeria',
    'HEALTHCARE INTERNATIONAL LIMITED': 'Nationwide – Nigeria',
    'HEALTHSPRING HMO': 'FCT',
    'HYGEIA HMO LIMITED': 'Nationwide – Nigeria',
    'Infinite X2 Health Maintenance Services': 'Kaduna',
    'INTEGRATED HEALTHCARE': 'Nationwide – Nigeria',
    'INTERNATIONAL HEALTH MGT. SERVICES': 'Nationwide – Nigeria',
    'IVES MEDICARE': 'Lagos',
    'KENNEDIA HMO LIMITED': 'Lagos',
    'LEADWAY HEALTH LIMITED': 'Nationwide – Nigeria',
    'LIFE WORTH MEDICARE LTD': 'Nationwide – Nigeria',
    'LIFESAVER HEALTHCARE LIMITED': 'Lagos',
    'MAAYOIT HEALTH CARE LIMITED': 'Kwara',
    'MARINA MEDICAL SERVICES HMO LIMITED': 'Lagos',
    'MARKFEMA NIGERIA LTD.': 'FCT',
    'MASSLIFE HEALTHCARE LIMITED': 'FCT',
    'MB & O HEALTHCARE SERVICES LIMITED': 'Lagos',
    'MEDEXIA LIMITED': 'Lagos',
    'MEDICARE ALLIANCE LIMITED': 'FCT',
    'MEDIPLAN HEALTHCARE LIMITED': 'Nationwide – Nigeria',
    'METROHEALTH HMO LIMITED': 'Nationwide – Nigeria',
    'NEM HEALTH LIMITED': 'Nationwide – Nigeria',
    'NNPC-HMO LIMITED': 'Nationwide – Nigeria',
    'NONSUCH MEDICARE LIMITED': 'Oyo',
    'NOOR HEALTH LTD.': 'Lagos',
    'NOVO HEALTH AFRICA LIMITED': 'Nationwide – Nigeria',
    'OCEANIC HEALTH MANAGEMENT LIMITED': 'Lagos / FCT',
    'PERAMARE HEALTH MANAGEMENT COMPANY LIMITED': 'FCT',
    'PHILLIPS HEALTH MANAGEMENT SERVICES LIMITED': 'Lagos',
    'POLICE HEALTH MAINTENANCE LIMITED': 'Nationwide – Nigeria',
    'PRECIOUS HEALTHCARE LIMITED': 'FCT',
    'PREPAID MEDICARE SERVICES LTD.': 'Nationwide – Nigeria',
    'PRINCETON HEALTH': 'Oyo',
    'PROHEALTH HMO LTD': 'Nationwide – Nigeria',
    'REDCARE HEALTH SERVICES LIMITED': 'Nationwide – Nigeria',
    'REGENIX HEALTH CARE SERVICE LIMITED': 'Rivers',
    'RELIANCE HMO LIMITED': 'Nationwide – Nigeria',
    'RODING HEALTHCARE LTD.': 'Lagos',
    'RONSBERGER NIGERIA LTD.': 'FCT',
    'ROTHAUGE HEALTHCARE LIMITED': 'Lagos',
    'ROYAL HEALTH MAINTENANCE SERVICES LTD.': 'Imo',
    'SALUS TRUST GTE': 'Lagos',
    'SERAPH HMO LIMITED': 'Benue',
    'SKYDA HEALTH LIMITED': 'FCT',
    'SONGHAI HEALTH TRUST': 'FCT',
    'SPRINGTIDE HEALTHCARE SERVICES LTD.': 'Rivers',
    'STERLING HEALTH MANAGED CARE SERVICES LIMITED': 'Nationwide – Nigeria',
    'SUNU HEALTH NIGERIA LIMITED': 'Nationwide – Nigeria',
    'SYNERGY WELLCARE MEDICAID LIMITED': 'Rivers',
    'TOTAL HEALTH TRUST LIMITED': 'Nationwide – Nigeria',
    'ULTIMATE HEALTH MANAGEMENT SERVICES LTD': 'Nationwide – Nigeria',
    'UNITED COMPREHENSIVE HEALTH MANAGERS LTD.': 'Rivers',
    'UNITED HEALTHCARE INTERNATIONAL LIMITED': 'Nationwide – Nigeria',
    'VENUS MEDICARE LTD': 'FCT',
    'VERITAS HEALTHCARE LIMITED': 'FCT',
    'WELL HEALTH NETWORK LIMITED': 'Enugu',
    'WELLNESS HEALTH MANAGEMENT SERVICES LIMITED': 'Lagos',
    'ZUMA HEALTH TRUST': 'FCT',
    'Healthnomics HMO PLC': 'FCT',
    'HYSSOP Health International Limited (HMO)': 'Delta',
    'Hopewell Healthcare Management Ltd': 'FCT',
    'Crown Jewel HMO Ltd.': 'FCT',
    'Smathealth Medicare Limited': 'Lagos',
    'Quest Medicare Limited': 'Lagos',
    'Life Action Plus Ltd.': 'Lagos',
    'Zenor Healthcare Ltd.': 'Lagos',
    'Aspire HMO Limited': 'Lagos',
    'Mitera Health Limited': 'Lagos'
}

nigeria_hmo['Coverage State'] = nigeria_hmo['Name'].map(coverage_map)

print("Missing Coverage State:", nigeria_hmo['Coverage State'].isna().sum())

nigeria_hmo[['Name', 'State', 'Coverage State']]

Missing Coverage State: 1


,Name,State,Coverage State
0,A&M HEALTHCARE TRUST LIMITED,Adamawa,Adamawa
1,AIICO MULTISHIELD NIGERIA LIMITED,Lagos,Nationwide – Nigeria
2,ALLEANZA HEALTH MANAGEMENT LIMITED,Lagos,Nationwide – Nigeria
3,ALLY HEALTHCARE LIMITED,FCT,FCT
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,FCT,FCT
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,N/A,Nationwide – Nigeria
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Kaduna,Kaduna
7,AVON HEALTHCARE LIMITED,Lagos,Nationwide – Nigeria
8,AXA MANSARD HEALTH LIMITED,Lagos,Nationwide – Nigeria
9,BASTION HEALTH LIMITED,Lagos,Lagos


In [100]:
nigeria_hmo['Coverage State'].value_counts(dropna=False)

Coverage State
Nationwide – Nigeria    35
Lagos                   24
FCT                     17
Rivers                   6
Kaduna                   2
Oyo                      2
Adamawa                  1
Kwara                    1
Lagos / FCT              1
Imo                      1
Benue                    1
Enugu                    1
Delta                    1
NaN                      1
Name: count, dtype: int64

In [101]:
nigeria_hmo[nigeria_hmo['Coverage State'].isna()][['S/NO.', 'Name', 'State', 'Website']]

,S/NO.,Name,State,Website
88,89,Health Assur Limited,Lagos,N/A


In [104]:
nigeria_hmo.loc[nigeria_hmo['Name'] == 'Health Assur Limited', 'Coverage State'] = 'Lagos'

In [106]:
nigeria_hmo['Coverage State'].isna().sum()

np.int64(0)

In [108]:
nigeria_hmo[['Name', 'Website', 'Coverage Type']]

,Name,Website,Coverage Type
0,A&M HEALTHCARE TRUST LIMITED,https://amhmo.com/,N/A
1,AIICO MULTISHIELD NIGERIA LIMITED,https://www.aiicomultishield.com/,N/A
2,ALLEANZA HEALTH MANAGEMENT LIMITED,https://alleanzahealth.com/individual-plans/,N/A
3,ALLY HEALTHCARE LIMITED,https://amanhmo.com/home,N/A
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,https://anchorhmo.com/,N/A
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,https://ashmedintegratedhealthservices.com/,N/A
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,https://www.avonhealthcare.com/,N/A
7,AVON HEALTHCARE LIMITED,https://www.axamansard.com/health/plans/,N/A
8,AXA MANSARD HEALTH LIMITED,https://bastionhmo.com/,N/A
9,BASTION HEALTH LIMITED,https://www.centurymedicaid.org/,N/A


In [111]:

coverage_type = {

    "A&M HEALTHCARE TRUST LIMITED":
        "Individual / Family / Corporate",

    "AIICO MULTISHIELD NIGERIA LIMITED":
        "Individual / Family / Corporate / SME / Group / International",

    "ALLEANZA HEALTH MANAGEMENT LIMITED":
        "Individual / Family / Corporate / Elderly / SME",

    "ALLY HEALTHCARE LIMITED":
        "Individual / Corporate",

    "AMAN HEALTH MAINTENANCE ORGANIZATIONS":
        "Individual / Family / Corporate",

    "ANCHOR HMO INTERNATIONAL COMPANY LIMITED":
        "Individual / Corporate / SME",

    "ASHMED INTEGRATED HEALTH SERVICES LTD.":
        "Individual / Family / Corporate",

    "AVON HEALTHCARE LIMITED":
        "Individual / Family / Corporate",

    "AXA MANSARD HEALTH LIMITED":
        "Individual / Family / Corporate",

    "BASTION HEALTH LIMITED":
        "Individual / Family / Corporate",

    "BONITAS HEALTH MAINTENANCE LIMITED":
        "Individual / Family / Corporate",

    "CENTURY MEDICAID SERVICES LIMITED":
        "Individual / Family / Corporate / Community / Industrial / Educational",

    "CLEARLINE INTERNATIONAL LIMITED":
        "Individual / Family / Corporate",

    "DEFENCE HEALTH MAINTENANCE LIMITED":
        "Corporate / Group / Military",

    "Delog Medical Services Ltd. (HMO)":
        "Individual / Family / Corporate",

    "DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO":
        "Individual / Family / Corporate",

    "DOT HMO Ltd.":
        "Individual / Family / Corporate",

    "FOUNTAIN HEALTHCARE LIMITED":
        "Individual / Family / Corporate",

    "GNI HEALTHCARE LTD":
        "Individual / Family / Corporate",

    "GORAH HEALTHCARE LTD.":
        "Individual / Family / Corporate",

    "GREENBAY HEALTHCARE SERVICES LIMITED":
        "Individual / Family / Corporate",

    "GREENFIELD HEALTH MANAGEMENT LTD":
        "Individual / Family / Corporate",

    "GROOMING HEALTH MANAGEMENT LIMITED":
        "Individual / Corporate / SME",

    "HALLMARK HEALTH SERVICES LIMITED":
        "Retail / SME / Corporate / International",

    "HEALTH ASSUR LIMITED":
        "Individual / Family / Business / Corporate",

    "Health Partners Limited":
        "Retail / SME / Corporate",

    "HEALTHCARE INTERNATIONAL LIMITED":
        "Individual / Family / Corporate / International",

    "HEALTHSPRING HMO":
        "Individual / Family",

    "HYGEIA HMO LIMITED":
        "Individual / Family / Senior Citizens / Maternity / SME / Corporate",

    "Infinite X2 Health Maintenance Services":
        "Individual / Family / Corporate",

    "INTEGRATED HEALTHCARE":
        "Individual / Family / Corporate",

    "INTERNATIONAL HEALTH MGT. SERVICES":
        "Corporate / Group",

    "IVES MEDICARE":
        "Individual / Family / Corporate",

    "KENNEDIA HMO LIMITED":
        "SME / Corporate / Group",

    "LEADWAY HEALTH LIMITED":
        "Retail / SME / Corporate / International",

    "LIFE WORTH MEDICARE LTD":
        "Individual / Family / Corporate",

    "LIFESAVER HEALTHCARE LIMITED":
        "Individual / Family / Corporate",

    "MAAYOIT HEALTH CARE LIMITED":
        "Individual / Family / Corporate",

    "MARINA MEDICAL SERVICES HMO LIMITED":
        "Individual / Family / Corporate",

    "MARKFEMA NIGERIA LTD.":
        "Individual / Family / Corporate",

    "MASSLIFE HEALTHCARE LIMITED":
        "Individual / Family / Corporate",

    "MB & O HEALTHCARE SERVICES LIMITED":
        "Individual / Family / Corporate",

    "MEDEXIA LIMITED":
        "Individual / Family / Corporate",

    "MEDICARE ALLIANCE LIMITED":
        "Individual / Family / Corporate",

    "MEDIPLAN HEALTHCARE LIMITED":
        "Corporate / Individual / Family / Diaspora",

    "METROHEALTH HMO LIMITED":
        "Individual / Family / Corporate",

    "NEM HEALTH LIMITED":
        "Individual / Family / SME / Corporate / Senior Citizens / Diaspora",

    "NNPC-HMO LIMITED":
        "Corporate / Group",

    "NONSUCH MEDICARE LIMITED":
        "Corporate / TISHIP / Community / NHIA",

    "NOOR HEALTH LTD.":
        "Individual / Family / Corporate",

    "NOVO HEALTH AFRICA LIMITED":
        "Individual / Family / Corporate",

    "OCEANIC HEALTH MANAGEMENT LIMITED":
        "Individual / Family / Corporate",

    "PERAMARE HEALTH MANAGEMENT COMPANY LIMITED":
        "Individual / Family / Corporate",

    "PHILLIPS HEALTH MANAGEMENT SERVICES LIMITED":
        "Individual / Family / Corporate",

    "POLICE HEALTH MAINTENANCE LIMITED":
        "Corporate / Group / Police",

    "PRECIOUS HEALTHCARE LIMITED":
        "Individual / Family / Corporate",

    "PREPAID MEDICARE SERVICES LTD.":
        "Individual / Family / Corporate",

    "PRINCETON HEALTH":
        "Individual / Family / Corporate",

    "PROHEALTH HMO LTD":
        "Individual / Family / Corporate / SME / Group / Social Health Insurance",

    "REDCARE HEALTH SERVICES LIMITED":
        "Individual / Family / Corporate",

    "REGENIX HEALTH CARE SERVICE LIMITED":
        "Individual / Family / Corporate",

    "RELIANCE HMO LIMITED":
        "Individual / Family / SME / Corporate",

    "RODING HEALTHCARE LTD.":
        "Individual / Family / Corporate",

    "RONSBERGER NIGERIA LTD.":
        "Individual / Family / Corporate",

    "ROTHAUGE HEALTHCARE LIMITED":
        "Individual / Family / Corporate",

    "ROYAL HEALTH MAINTENANCE SERVICES LTD.":
        "Individual / Family / Corporate",

    "SALUS TRUST GTE":
        "Individual / Family / Student / Corporate",

    "SERAPH HMO LIMITED":
        "Individual / Family / Corporate",

    "SKYDA HEALTH LIMITED":
        "Individual / Family / Corporate",

    "SONGHAI HEALTH TRUST":
        "Individual / Family / Corporate",

    "SPRINGTIDE HEALTHCARE SERVICES LTD.":
        "Individual / Corporate / International",

    "STERLING HEALTH MANAGED CARE SERVICES LIMITED":
        "Individual / Family / Corporate",

    "SUNU HEALTH NIGERIA LIMITED":
        "Individual / Family / Employee / Corporate",

    "SYNERGY WELLCARE MEDICAID LIMITED":
        "Individual / Family / Employee / Business",

    "TOTAL HEALTH TRUST LIMITED":
        "Individual / Family / Corporate",

    "ULTIMATE HEALTH MANAGEMENT SERVICES LTD":
        "Group / Individual / Family / SME / Corporate",

    "UNITED COMPREHENSIVE HEALTH MANAGERS LTD.":
        "Individual / Family / Corporate",

    "UNITED HEALTHCARE INTERNATIONAL LIMITED":
        "Retail / Individual / Family / Corporate / International / Travel",

    "VENUS MEDICARE LTD":
        "Individual / Small Employer / Corporate",

    "VERITAS HEALTHCARE LIMITED":
        "Retail / Individual / Family / Corporate / TISHIP",

    "WELL HEALTH NETWORK LIMITED":
        "Individual / Family / Corporate / Group / Institutional / Student",

    "WELLNESS HEALTH MANAGEMENT SERVICES LIMITED":
        "Individual / Family / Corporate",

    "ZUMA HEALTH TRUST":
        "Individual / Family / Corporate",

    "Healthnomics HMO PLC":
        "Individual / Family",

    "HYSSOP Health International Limited (HMO)":
        "Individual / Family / Corporate",

    "Hopewell Healthcare Management Ltd":
        "Individual / Family",

    "Crown Jewel HMO Ltd.":
        "Individual / Family / SME / Corporate",

    "Smathealth Medicare Limited":
        "Individual / Family / Corporate",

    "Quest Medicare Limited":
        "Individual / Family / Corporate",

    "Life Action Plus Ltd.":
        "Individual / Family / Retail / SME / Corporate / Retirement / Community",

    "Zenor Healthcare Ltd.":
        "Individual / Family / SME / Corporate",

    "Aspire HMO Limited":
        "Individual / Family / Corporate",

    "Mitera Health Limited":
        "Individual / Family / Corporate"
}


# Apply the mapping
nigeria_hmo["Coverage Type"] = nigeria_hmo["Name"].map(coverage_type)


# ============================================================
# CHECKS
# ============================================================

print("Total rows:", len(nigeria_hmo))
print("Missing Coverage Type:", nigeria_hmo["Coverage Type"].isna().sum())

# Display the completed column
display(
    nigeria_hmo[["Name", "Coverage Type"]]
)


# Check the unique categories used
print("\nCoverage Type counts:")
print(nigeria_hmo["Coverage Type"].value_counts())

Total rows: 94
Missing Coverage Type: 1


,Name,Coverage Type
0,A&M HEALTHCARE TRUST LIMITED,Individual / Family / Corporate
1,AIICO MULTISHIELD NIGERIA LIMITED,Individual / Family / Corporate / SME / Group / International
2,ALLEANZA HEALTH MANAGEMENT LIMITED,Individual / Family / Corporate / Elderly / SME
3,ALLY HEALTHCARE LIMITED,Individual / Corporate
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,Individual / Family / Corporate
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,Individual / Corporate / SME
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Individual / Family / Corporate
7,AVON HEALTHCARE LIMITED,Individual / Family / Corporate
8,AXA MANSARD HEALTH LIMITED,Individual / Family / Corporate
9,BASTION HEALTH LIMITED,Individual / Family / Corporate



Coverage Type counts:
Coverage Type
Individual / Family / Corporate                                            56
Individual / Family / SME / Corporate                                       3
Individual / Family                                                         3
Individual / Corporate / SME                                                2
Corporate / Group                                                           2
Retail / SME / Corporate / International                                    2
Individual / Family / Employee / Corporate                                  1
Individual / Family / Student / Corporate                                   1
Individual / Corporate / International                                      1
Group / Individual / Family / SME / Corporate                               1
Individual / Family / Employee / Business                                   1
Corporate / Group / Police                                                  1
Retail / Individual / Famil

In [112]:

services = {

"A&M HEALTHCARE TRUST LIMITED":
"Outpatient care; Specialist consultation; Inpatient care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"AIICO MULTISHIELD NIGERIA LIMITED":
"Medical consultation; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Hospital care; Emergency care; Maternity care",

"ALLEANZA HEALTH MANAGEMENT LIMITED":
"Medical consultation; Hospital admission; Surgery; Maternity care; Emergency care; Pharmacy services; Provider network access",

"ALLY HEALTHCARE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"AMAN HEALTH MAINTENANCE ORGANIZATIONS":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"ANCHOR HMO INTERNATIONAL COMPANY LIMITED":
"Corporate health plans; Retail health plans; Third-party administration; Customized health plans; Emergency support; Claims management",

"ASHMED INTEGRATED HEALTH SERVICES LTD.":
"Managed healthcare; Health insurance; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"AVON HEALTHCARE LIMITED":
"Health insurance; Outpatient care; Inpatient care; Specialist consultation; Diagnostic services; Prescription medicines; Emergency care; Maternity care; Dental care; Optical care",

"AXA MANSARD HEALTH LIMITED":
"Outpatient care; Inpatient care; Surgical services; Routine immunization; Emergency evacuation; Dental care; Eye care; Physiotherapy; Telemedicine; Pharmacy services; Hospital cash",

"BASTION HEALTH LIMITED":
"Health insurance; Outpatient care; Inpatient care; Specialist care; Hospital services; Emergency care; Provider network access; Corporate and individual health plans",

"BONITAS HEALTH MAINTENANCE LIMITED":
"Health insurance; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"CENTURY MEDICAID SERVICES LIMITED":
"Managed healthcare; Health insurance; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"CLEARLINE INTERNATIONAL LIMITED":
"Outpatient care; General and specialist consultation; Teleconsultation; Prescription medicines; Diagnostic and laboratory services; Hospital care; Emergency care",

"DEFENCE HEALTH MAINTENANCE LIMITED":
"Military and defence health insurance; Primary healthcare; Secondary healthcare; Tertiary healthcare; Clinical services; Pharmacy services; Nursing care; Allied medical services; Public health services",

"Delog Medical Services Ltd. (HMO)":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"DOT HMO Ltd.":
"Clinic visits; Diagnostic services; Prescription medicines; Specialist care; Telemedicine; Wellness and preventive care; Emergency support",

"FOUNTAIN HEALTHCARE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"GNI HEALTHCARE LTD":
"Medical consultation; Specialist care; Laboratory and diagnostic services; Prescription medicines; Maternity care; Eye care; Dental care; Immunization; Hospital admission; Surgery; HIV care; Ambulance services",

"GORAH HEALTHCARE LTD.":
"Preventive healthcare; Wellness care; Emergency services; Specialist consultation; Maternity care; Hospital and clinic access",

"GREENBAY HEALTHCARE SERVICES LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"GREENFIELD HEALTH MANAGEMENT LTD":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"GROOMING HEALTH MANAGEMENT LIMITED":
"Routine checkups; Specialist consultation; Emergency care; Outpatient care; Inpatient care; Health insurance; Corporate and individual health plans",

"HALLMARK HEALTH SERVICES LIMITED":
"General consultation; Specialist consultation; Diagnostic investigations; Physiotherapy; Renal dialysis; Surgical care; Telemedicine; Gym access; Mortuary cover; Hospital care",

"HEALTH ASSUR LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"Health Partners Limited":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"HEALTHCARE INTERNATIONAL LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"HEALTHSPRING HMO":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"HYGEIA HMO LIMITED":
"Outpatient care; Inpatient care; Optical care; Dental care; Chronic disease management; Telemedicine; Physiotherapy; ENT services; Wellness and preventive care; Maternal and child care",

"Infinite X2 Health Maintenance Services":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"INTEGRATED HEALTHCARE":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"INTERNATIONAL HEALTH MGT. SERVICES":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"IVES MEDICARE":
"Health insurance; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"KENNEDIA HMO LIMITED":
"Managed healthcare; Medical consultation; Doctor consultation; Hospital appointments; Provider network access; Health insurance plans; Telemedicine",

"LEADWAY HEALTH LIMITED":
"Outpatient care; Inpatient care; Optical care; Dental care; Telemedicine; Emergency services; Mental health care; Chronic disease management; Pharmacy services; Home vaccination; Fitness and nutrition services",

"LIFE WORTH MEDICARE LTD":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"LIFESAVER HEALTHCARE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"MAAYOIT HEALTH CARE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"MARINA MEDICAL SERVICES HMO LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"MARKFEMA NIGERIA LTD.":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"MASSLIFE HEALTHCARE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"MB & O HEALTHCARE SERVICES LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"MEDEXIA LIMITED":
"Prepaid health plans; Third-party administration; Clinic management; Health audit; Health, safety and environment consultancy; Medical expense administration",

"MEDICARE ALLIANCE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"MEDIPLAN HEALTHCARE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"METROHEALTH HMO LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"NEM HEALTH LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"NNPC-HMO LIMITED":
"Employee health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"NONSUCH MEDICARE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"NOOR HEALTH LTD.":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"NOVO HEALTH AFRICA LIMITED":
"Quality healthcare; Health insurance plans; Community health project management; Healthcare financing and administration; Health system support; Retail health plans; Geriatric care; Corporate health plans",

"OCEANIC HEALTH MANAGEMENT LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"PERAMARE HEALTH MANAGEMENT COMPANY LIMITED":
"Emergency care; Outpatient care; Diagnostic services; Prescription medicines; Physiotherapy; Advanced investigations; Inpatient care; Surgery; Maternity care; Dental care; Eye care; Child welfare; Immunization; Mental health therapy; Annual medical checks; Home healthcare; Gym and spa access; Teleconsultation",

"PHILLIPS HEALTH MANAGEMENT SERVICES LIMITED":
"Health insurance; Outpatient care; Inpatient care; Specialist care; Surgical care; Pre-employment screening; Annual medical checkups; Hospital provider network",

"POLICE HEALTH MAINTENANCE LIMITED":
"Police personnel health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"PRECIOUS HEALTHCARE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"PREPAID MEDICARE SERVICES LTD.":
"Prepaid health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"PRINCETON HEALTH":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"PROHEALTH HMO LTD":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"REDCARE HEALTH SERVICES LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"REGENIX HEALTH CARE SERVICE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"RELIANCE HMO LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Telemedicine",

"RODING HEALTHCARE LTD.":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"RONSBERGER NIGERIA LTD.":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"ROTHAUGE HEALTHCARE LIMITED":
"Specialist consultation; Outpatient services; Hospital admission; Prescription drugs; Immunization; Laboratory investigations; Physiotherapy; Dental care; Optical care; Surgical procedures; Maternity care; ICU services; Preventive wellness; Employment screening; Third-party administration; Healthcare consulting",

"ROYAL HEALTH MAINTENANCE SERVICES LTD.":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"SALUS TRUST GTE":
"Health insurance; Routine consultation; Specialist treatment; Health screening; Diagnostic testing; Pharmacy services; Chronic illness management; Wellness programmes; Fitness and nutrition guidance; Counselling; Preventive care; Health education",

"SERAPH HMO LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"SKYDA HEALTH LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"SONGHAI HEALTH TRUST":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"SPRINGTIDE HEALTHCARE SERVICES LTD.":
"Outpatient care; General consultation; Specialist consultation; Inpatient care; Hospital access; Prescription medicines; Chronic disease management; Diagnostic services; Laboratory services; Radiology; Eye care; Dental care; Wellness programmes; Preventive care; Telemedicine; Health coaching",

"STERLING HEALTH MANAGED CARE SERVICES LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"SUNU HEALTH NIGERIA LIMITED":
"Managed healthcare; Health insurance; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Telemedicine; Overseas referral services",

"SYNERGY WELLCARE MEDICAID LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"TOTAL HEALTH TRUST LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Maternity care",

"ULTIMATE HEALTH MANAGEMENT SERVICES LTD":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"UNITED COMPREHENSIVE HEALTH MANAGERS LTD.":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"UNITED HEALTHCARE INTERNATIONAL LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"VENUS MEDICARE LTD":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"VERITAS HEALTHCARE LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"WELL HEALTH NETWORK LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"WELLNESS HEALTH MANAGEMENT SERVICES LIMITED":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care; Wellness and preventive care",

"ZUMA HEALTH TRUST":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"Healthnomics HMO PLC":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"HYSSOP Health International Limited (HMO)":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"Hopewell Healthcare Management Ltd":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"Crown Jewel HMO Ltd.":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"Smathealth Medicare Limited":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"Quest Medicare Limited":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"Life Action Plus Ltd.":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"Zenor Healthcare Ltd.":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"Aspire HMO Limited":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care",

"Mitera Health Limited":
"Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care"
}


nigeria_hmo["Services Rendered"] = nigeria_hmo["Name"].map(services)

print("Missing Services:", nigeria_hmo["Services Rendered"].isna().sum())

nigeria_hmo[["Name", "Services Rendered"]]

Missing Services: 1


,Name,Services Rendered
0,A&M HEALTHCARE TRUST LIMITED,Outpatient care; Specialist consultation; Inpatient care; Diagnostic services; Prescription medicines; Emergency care; Maternity care
1,AIICO MULTISHIELD NIGERIA LIMITED,Medical consultation; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Hospital care; Emergency care; Maternity care
2,ALLEANZA HEALTH MANAGEMENT LIMITED,Medical consultation; Hospital admission; Surgery; Maternity care; Emergency care; Pharmacy services; Provider network access
3,ALLY HEALTHCARE LIMITED,Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,Corporate health plans; Retail health plans; Third-party administration; Customized health plans; Emergency support; Claims management
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Managed healthcare; Health insurance; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care
7,AVON HEALTHCARE LIMITED,Health insurance; Outpatient care; Inpatient care; Specialist consultation; Diagnostic services; Prescription medicines; Emergency care; Maternity care; Dental care; Optical care
8,AXA MANSARD HEALTH LIMITED,Outpatient care; Inpatient care; Surgical services; Routine immunization; Emergency evacuation; Dental care; Eye care; Physiotherapy; Telemedicine; Pharmacy services; Hospital cash
9,BASTION HEALTH LIMITED,Health insurance; Outpatient care; Inpatient care; Specialist care; Hospital services; Emergency care; Provider network access; Corporate and individual health plans


In [114]:
nigeria_hmo[nigeria_hmo["Services Rendered"].isna()][["S/NO.", "Name", "Website"]]

,S/NO.,Name,Website
88,89,Health Assur Limited,N/A


In [118]:
nigeria_hmo = nigeria_hmo.drop(index=88).reset_index(drop=True)

print("Number of rows:", len(nigeria_hmo))
print(nigeria_hmo[nigeria_hmo["Name"].str.strip().str.lower() == "health assur limited"])

Number of rows: 91
    S/NO.                  Name  HMO ID                        Website  \
24     25  HEALTH ASSUR LIMITED      87  https://www.hcihealthcare.ng/   

                                          Address                 Email  \
24  NO.1 RAMAT CRESCENT OGUDU GRA, KOSOFE, LAGOS.  info@healthassur.com   

                                                 Contact Number  Country  \
24  01-3422321, 08026547348, 0700 HEALTH ASSUR (07004325427787)  Nigeria   

   Source                    Source URL  State      Region  \
24   NHIA  https://www.nhia.gov.ng/hmo/  Lagos  South West   

          Coverage State                               Coverage Type  \
24  Nationwide – Nigeria  Individual / Family / Business / Corporate   

                                                                                                                                                      Services Rendered  \
24  Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist c

In [119]:
print("Missing Services:", nigeria_hmo["Services Rendered"].isna().sum())
print("Missing Coverage Type:", nigeria_hmo["Coverage Type"].isna().sum())

Missing Services: 0
Missing Coverage Type: 0


In [120]:
target_customers = {

"A&M HEALTHCARE TRUST LIMITED":
"Individuals; Families; Corporate Organisations",

"AIICO MULTISHIELD NIGERIA LIMITED":
"Individuals; Families; SMEs; Corporate Organisations; Associations; Large Groups",

"ALLEANZA HEALTH MANAGEMENT LIMITED":
"Individuals; Families; Elderly/Senior Citizens; SMEs; Corporate Organisations",

"ALLY HEALTHCARE LIMITED":
"Individuals; Families; Corporate Organisations",

"AMAN HEALTH MAINTENANCE ORGANIZATIONS":
"Individuals; Families; Corporate Organisations",

"ANCHOR HMO INTERNATIONAL COMPANY LIMITED":
"Individuals; Families; SMEs; Small and Large Employers; Corporate Organisations",

"ASHMED INTEGRATED HEALTH SERVICES LTD.":
"Individuals; Families; Corporate Organisations",

"AVON HEALTHCARE LIMITED":
"Individuals; Couples; Families; Businesses; Corporate Organisations",

"AXA MANSARD HEALTH LIMITED":
"Individuals; Families; Employees; Groups; Corporate Organisations",

"BASTION HEALTH LIMITED":
"Individuals; Families; Senior Citizens; SMEs; Corporate Organisations",

"BONITAS HEALTH MAINTENANCE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"CENTURY MEDICAID SERVICES LIMITED":
"Individuals; Families; Corporate Organisations; Employees; Students; Educational Institutions; Industry Groups",

"CLEARLINE INTERNATIONAL LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"DEFENCE HEALTH MAINTENANCE LIMITED":
"Defence Personnel; Defence Employees; Dependants",

"Delog Medical Services Ltd. (HMO)":
"Individuals; Families; SMEs; Corporate Organisations",

"DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO":
"Individuals; Families; SMEs; Corporate Organisations",

"DOT HMO Ltd.":
"Individuals; Families; SMEs; Teams",

"FOUNTAIN HEALTHCARE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"GNI HEALTHCARE LTD":
"Individuals; Families; Employees; Corporate Organisations",

"GORAH HEALTHCARE LTD.":
"Individuals; Families; Students; Senior Citizens; Corporate Organisations; International Clients",

"GREENBAY HEALTHCARE SERVICES LIMITED":
"Individuals; Families; Employers; Businesses; Corporate Organisations",

"GREENFIELD HEALTH MANAGEMENT LTD":
"Individuals; Families; SMEs; Corporate Organisations",

"GROOMING HEALTH MANAGEMENT LIMITED":
"Individuals; Families; Employees; Corporate Organisations",

"HALLMARK HEALTH SERVICES LIMITED":
"Individuals; Families; Employees; Corporate Organisations",

"HEALTH ASSUR LIMITED":
"Individuals; Families; Employees; SMEs; Corporate Organisations",

"Health Partners Limited":
"Individuals; Families; SMEs; Corporate Organisations",

"HEALTHCARE INTERNATIONAL LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"HEALTHSPRING HMO":
"Individuals; Families",

"HYGEIA HMO LIMITED":
"Individuals; Families; Senior Citizens; SMEs; Corporate Organisations",

"Infinite X2 Health Maintenance Services":
"Individuals; Families; Corporate Organisations",

"INTEGRATED HEALTHCARE":
"Individuals; Families; Corporate Organisations",

"INTERNATIONAL HEALTH MGT. SERVICES":
"Individuals; Families; Corporate Organisations",

"IVES MEDICARE":
"Individuals; Families; Corporate Organisations",

"KENNEDIA HMO LIMITED":
"SMEs; Corporate Organisations; Large Corporates",

"LEADWAY HEALTH LIMITED":
"Individuals; Families; Senior Citizens; SMEs; Small and Large Companies; International Clients",

"LIFE WORTH MEDICARE LTD":
"Individuals; Families; SMEs; Corporate Organisations",

"LIFESAVER HEALTHCARE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"MAAYOIT HEALTH CARE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"MARINA MEDICAL SERVICES HMO LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"MARKFEMA NIGERIA LTD.":
"Individuals; Families; Organisations; Companies; Government Bodies",

"MASSLIFE HEALTHCARE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations; Senior Citizens; Tertiary Students; MSMEs",

"MB & O HEALTHCARE SERVICES LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"MEDEXIA LIMITED":
"Healthcare Organisations; Healthcare Providers; Corporate Organisations",

"MEDICARE ALLIANCE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"MEDIPLAN HEALTHCARE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"METROHEALTH HMO LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"NEM HEALTH LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"NNPC-HMO LIMITED":
"NNPC Employees; NNPC Dependants; Employees of Participating Organisations",

"NONSUCH MEDICARE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"NOOR HEALTH LTD.":
"Individuals; Families; SMEs; Corporate Organisations",

"NOVO HEALTH AFRICA LIMITED":
"Individuals; Families; Employers; Corporate Organisations; Government Organisations; Communities",

"OCEANIC HEALTH MANAGEMENT LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"PERAMARE HEALTH MANAGEMENT COMPANY LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"PHILLIPS HEALTH MANAGEMENT SERVICES LIMITED":
"Individuals; Families; Employees; Corporate Organisations",

"POLICE HEALTH MAINTENANCE LIMITED":
"Police Personnel; Police Employees; Dependants",

"PRECIOUS HEALTHCARE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"PREPAID MEDICARE SERVICES LTD.":
"Individuals; Families; SMEs; Corporate Organisations",

"PRINCETON HEALTH":
"Individuals; Families; SMEs; Corporate Organisations",

"PROHEALTH HMO LTD":
"Individuals; Families; SMEs; Corporate Organisations",

"REDCARE HEALTH SERVICES LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"REGENIX HEALTH CARE SERVICE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"RELIANCE HMO LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"RODING HEALTHCARE LTD.":
"Individuals; Families; SMEs; Corporate Organisations",

"RONSBERGER NIGERIA LTD.":
"Individuals; Families; SMEs; Corporate Organisations",

"ROTHAUGE HEALTHCARE LIMITED":
"Individuals; Families; Employees; Employers; Corporate Organisations",

"ROYAL HEALTH MAINTENANCE SERVICES LTD.":
"Individuals; Families; SMEs; Corporate Organisations",

"SALUS TRUST GTE":
"Individuals; Families; Employees; Corporate Organisations",

"SERAPH HMO LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"SKYDA HEALTH LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"SONGHAI HEALTH TRUST":
"Individuals; Families; SMEs; Corporate Organisations",

"SPRINGTIDE HEALTHCARE SERVICES LTD.":
"Individuals; Families; Employees; Corporate Organisations",

"STERLING HEALTH MANAGED CARE SERVICES LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"SUNU HEALTH NIGERIA LIMITED":
"Individuals; Families; SMEs; Corporate Organisations; Employees",

"SYNERGY WELLCARE MEDICAID LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"TOTAL HEALTH TRUST LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"ULTIMATE HEALTH MANAGEMENT SERVICES LTD":
"Individuals; Families; SMEs; Corporate Organisations",

"UNITED COMPREHENSIVE HEALTH MANAGERS LTD.":
"Individuals; Families; SMEs; Corporate Organisations",

"UNITED HEALTHCARE INTERNATIONAL LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"VENUS MEDICARE LTD":
"Individuals; Families; SMEs; Corporate Organisations",

"VERITAS HEALTHCARE LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"WELL HEALTH NETWORK LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"WELLNESS HEALTH MANAGEMENT SERVICES LIMITED":
"Individuals; Families; SMEs; Corporate Organisations",

"ZUMA HEALTH TRUST":
"Individuals; Families; SMEs; Corporate Organisations",

"Healthnomics HMO PLC":
"Individuals; Families; SMEs; Corporate Organisations",

"HYSSOP Health International Limited (HMO)":
"Individuals; Families; SMEs; Corporate Organisations",

"Hopewell Healthcare Management Ltd":
"Individuals; Families; SMEs; Corporate Organisations",

"Crown Jewel HMO Ltd.":
"Individuals; Families; SMEs; Corporate Organisations",

"Smathealth Medicare Limited":
"Individuals; Families; SMEs; Corporate Organisations",

"Quest Medicare Limited":
"Individuals; Families; SMEs; Corporate Organisations",

"Life Action Plus Ltd.":
"Individuals; Families; SMEs; Corporate Organisations",

"Zenor Healthcare Ltd.":
"Individuals; Families; SMEs; Corporate Organisations",

"Aspire HMO Limited":
"Individuals; Families; SMEs; Corporate Organisations",

"Mitera Health Limited":
"Individuals; Families; SMEs; Corporate Organisations"
}


nigeria_hmo["Target Customers"] = nigeria_hmo["Name"].map(target_customers)


print("Missing Target Customers:",
      nigeria_hmo["Target Customers"].isna().sum())

print("\nNumber of rows:",
      len(nigeria_hmo))

nigeria_hmo[["Name", "Target Customers"]]

Missing Target Customers: 0

Number of rows: 91


,Name,Target Customers
0,A&M HEALTHCARE TRUST LIMITED,Individuals; Families; Corporate Organisations
1,AIICO MULTISHIELD NIGERIA LIMITED,Individuals; Families; SMEs; Corporate Organisations; Associations; Large Groups
2,ALLEANZA HEALTH MANAGEMENT LIMITED,Individuals; Families; Elderly/Senior Citizens; SMEs; Corporate Organisations
3,ALLY HEALTHCARE LIMITED,Individuals; Families; Corporate Organisations
4,AMAN HEALTH MAINTENANCE ORGANIZATIONS,Individuals; Families; Corporate Organisations
5,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,Individuals; Families; SMEs; Small and Large Employers; Corporate Organisations
6,ASHMED INTEGRATED HEALTH SERVICES LTD.,Individuals; Families; Corporate Organisations
7,AVON HEALTHCARE LIMITED,Individuals; Couples; Families; Businesses; Corporate Organisations
8,AXA MANSARD HEALTH LIMITED,Individuals; Families; Employees; Groups; Corporate Organisations
9,BASTION HEALTH LIMITED,Individuals; Families; Senior Citizens; SMEs; Corporate Organisations


In [121]:
combined_df = pd.concat(
    [nigeria_hmo, england_wales_df],
    ignore_index=True
)

combined_df

,S/NO.,Name,HMO ID,Website,Address,Email,Contact Number,Country,Source,Source URL,State,Region,Coverage State,Coverage Type,Services Rendered,Target Customers
0,1.0,A&M HEALTHCARE TRUST LIMITED,102.0,https://amhmo.com/,"Plot U Bekaji Road, Opposite Bekaji Jumma'at Mosque, Jimeta, Yola, Adamawa State",info@amhmo.com,"08033646497, 09162788582, 08024143666",Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,Adamawa,North East,Adamawa,Individual / Family / Corporate,Outpatient care; Specialist consultation; Inpatient care; Diagnostic services; Prescription medicines; Emergency care; Maternity care,Individuals; Families; Corporate Organisations
1,2.0,AIICO MULTISHIELD NIGERIA LIMITED,6.0,https://www.aiicomultishield.com/,"322, IKORODU ROAD, ANTHONY",info@aiicomultishield.com loshunniyi@aiicomultishield.com,07026744353 08056744353,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,Lagos,South West,Nationwide – Nigeria,Individual / Family / Corporate / SME / Group / International,Medical consultation; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Hospital care; Emergency care; Maternity care,Individuals; Families; SMEs; Corporate Organisations; Associations; Large Groups
2,3.0,ALLEANZA HEALTH MANAGEMENT LIMITED,111.0,https://alleanzahealth.com/individual-plans/,"? 83B, Basheer Shittu Avenue, Magodo Estate, Phase 2, Shangisha, Lagos",info@alleanzahealth.com,07036592835,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,Lagos,South West,Nationwide – Nigeria,Individual / Family / Corporate / Elderly / SME,Medical consultation; Hospital admission; Surgery; Maternity care; Emergency care; Pharmacy services; Provider network access,Individuals; Families; Elderly/Senior Citizens; SMEs; Corporate Organisations
3,4.0,ALLY HEALTHCARE LIMITED,119.0,https://amanhmo.com/home,"Suit C4 Plot 1196 Ndjamena Crescent, Wuse II, Abuja",allyhealthcarelimited@gmail.com,08148808171,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,FCT,North Central,FCT,Individual / Corporate,Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care,Individuals; Families; Corporate Organisations
4,5.0,AMAN HEALTH MAINTENANCE ORGANIZATIONS,121.0,https://anchorhmo.com/,"? 1, M. M. Alkali Street, Off 442 Crescent, Citec Vilas Gwarinpa, Abuja",info@amanhmo.com,08100588906,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,FCT,North Central,FCT,Individual / Family / Corporate,Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care,Individuals; Families; Corporate Organisations
5,6.0,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,76.0,https://ashmedintegratedhealthservices.com/,GOLDCREST MALL BUILDING,info@anchorhmo.com,"08031230306, 07058890062, 07080601192",Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,N/A,N/A,Nationwide – Nigeria,Individual / Corporate / SME,Corporate health plans; Retail health plans; Third-party administration; Customized health plans; Emergency support; Claims management,Individuals; Families; SMEs; Small and Large Employers; Corporate Organisations
6,7.0,ASHMED INTEGRATED HEALTH SERVICES LTD.,89.0,https://www.avonhealthcare.com/,"RE 013 ZUNGERU CLOSE BY BIMA ROAD NITEL QUARTERS, KADUNA",info@ashmedintegratedhealthservices.com,07036035182,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,Kaduna,North West,Kaduna,Individual / Family / Corporate,Managed healthcare; Health insurance; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care,Individuals; Families; Corporate Organisations
7,8.0,AVON HEALTHCARE LIMITED,63.0,https://www.axamansard.com/health/plans/,"22B, GLOVER ROAD",info@avonhealthcare.com,"08102659972, 07002779800",Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,Lagos,South West,Nationwide – Nigeria,Individual / Family / Corporate,Health insurance; Outpatient care; Inpatient care; Specialist consultation; Diagnostic services; Prescript

In [123]:
%whos DataFrame

Variable           Type         Data/Info
-----------------------------------------
combined_df        DataFrame    Shape: (117, 16)
coverage_na        DataFrame    Shape: (67, 3)
england_df         DataFrame    Shape: (14, 13)
england_wales_df   DataFrame    Shape: (26, 13)
health_df          DataFrame    Shape: (14, 13)
nigeria_df         DataFrame    Shape: (0, 13)
nigeria_hmo        DataFrame    Shape: (91, 16)
wales_df           DataFrame    Shape: (12, 13)
website_df         DataFrame    Shape: (94, 2)


In [124]:
df = combined_df.copy()

print("========================================")
print("DATASET OVERVIEW")
print("========================================")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
print()

print("COLUMNS:")
print(df.columns.tolist())
print()

print("========================================")
print("MISSING VALUES (NaN)")
print("========================================")
print(df.isna().sum())
print()

print("========================================")
print("N/A VALUES")
print("========================================")
na_counts = (df.astype(str).apply(
    lambda col: col.str.strip().str.upper().eq("N/A")
)).sum()

print(na_counts)
print()

print("========================================")
print("DUPLICATE ORGANISATION NAMES")
print("========================================")

duplicates = df[df["Name"].duplicated(keep=False)].sort_values("Name")

if len(duplicates) == 0:
    print("No duplicate organisation names found.")
else:
    print(duplicates[["Name", "Country", "Website"]].to_string(index=False))

print()

print("========================================")
print("DUPLICATE WEBSITES")
print("========================================")

website_duplicates = df[
    df["Website"].duplicated(keep=False) &
    df["Website"].notna() &
    (df["Website"].astype(str).str.upper() != "N/A")
].sort_values("Website")

if len(website_duplicates) == 0:
    print("No duplicate websites found.")
else:
    print(
        website_duplicates[
            ["Name", "Website", "Country"]
        ].to_string(index=False)
    )

print()

print("========================================")
print("COUNTRY COUNTS")
print("========================================")
print(df["Country"].value_counts(dropna=False))
print()

print("========================================")
print("STATE COUNTS")
print("========================================")
print(df["State"].value_counts(dropna=False))
print()

print("========================================")
print("REGION COUNTS")
print("========================================")
print(df["Region"].value_counts(dropna=False))
print()

print("========================================")
print("COVERAGE STATE COUNTS")
print("========================================")
print(df["Coverage State"].value_counts(dropna=False))
print()

print("========================================")
print("S/NO. CHECK")
print("========================================")

print("Missing S/NO.:", df["S/NO."].isna().sum())
print("Duplicate S/NO.:", df["S/NO."].duplicated().sum())
print()

print("========================================")
print("COMPLETELY DUPLICATED ROWS")
print("========================================")

print("Number of completely duplicated rows:",
      df.duplicated().sum())

print()

print("========================================")
print("ROWS CONTAINING NaN")
print("========================================")

rows_with_nan = df[df.isna().any(axis=1)]

if len(rows_with_nan) == 0:
    print("No rows contain NaN.")
else:
    print(rows_with_nan.to_string(index=False))

print()

print("========================================")
print("QUALITY CHECK COMPLETE")
print("========================================")

DATASET OVERVIEW
Number of rows: 117
Number of columns: 16

COLUMNS:
['S/NO.', 'Name', 'HMO ID', 'Website', 'Address', 'Email', 'Contact Number', 'Country', 'Source', 'Source URL', 'State', 'Region', 'Coverage State', 'Coverage Type', 'Services Rendered', 'Target Customers']

MISSING VALUES (NaN)
S/NO.                26
Name                  0
HMO ID               26
Website               0
Address              27
Email                 0
Contact Number        0
Country               0
Source                0
Source URL            0
State                 0
Region                0
Coverage State        0
Coverage Type         0
Services Rendered     0
Target Customers      0
dtype: int64

N/A VALUES
S/NO.                 0
Name                  0
HMO ID                0
Website               4
Address               0
Email                18
Contact Number       12
Country               0
Source                0
Source URL            0
State                 5
Region               32
Cover

In [125]:
import pandas as pd

final_dataset = combined_df.copy()

website_corrections = {
    "Health Partners Limited": "https://healthpartnersng.org/",
    "MAAYOIT HEALTH CARE LIMITED": "N/A",
    "MB & O HEALTHCARE SERVICES LIMITED": "https://www.mbandohmo.com/",
    "NOOR HEALTH LTD.": "https://www.noorhealth.ng/",
    "SALUS TRUST GTE": "https://www.salustrustng.com/",
    "DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO": "https://doheec.com/",
    "Zenor Healthcare Ltd.": "https://www.zenorhealthcare.com/",
    "Mitera Health Limited": "https://miterahealth.com/"
}

for organisation, website in website_corrections.items():
    final_dataset.loc[
        final_dataset["Name"].str.strip().str.upper()
        == organisation.strip().upper(),
        "Website"
    ] = website

final_dataset["Website"] = (
    final_dataset["Website"]
    .astype(str)
    .str.strip()
)

final_dataset["Website"] = final_dataset["Website"].replace(
    ["nan", "NaN", ""], "N/A"
)

final_dataset["S/NO."] = range(1, len(final_dataset) + 1)

final_dataset.loc[
    final_dataset["Country"] == "United Kingdom",
    "HMO ID"
] = "N/A"

final_dataset = final_dataset.fillna("N/A")

before = len(final_dataset)

final_dataset = final_dataset.drop_duplicates(
    subset=["Name"],
    keep="first"
).reset_index(drop=True)

after = len(final_dataset)

print("Rows before duplicate check:", before)
print("Rows after duplicate check:", after)

final_dataset["S/NO."] = range(1, len(final_dataset) + 1)

duplicate_names = final_dataset[
    final_dataset["Name"].duplicated(keep=False)
]

print("\n========================================")
print("DUPLICATE NAME CHECK")
print("========================================")

if duplicate_names.empty:
    print("No duplicate organisation names found.")
else:
    print(duplicate_names[["S/NO.", "Name", "Country"]])

valid_websites = final_dataset[
    final_dataset["Website"].str.upper() != "N/A"
]

duplicate_websites = valid_websites[
    valid_websites["Website"].duplicated(keep=False)
].sort_values("Website")

print("\n========================================")
print("DUPLICATE WEBSITE CHECK")
print("========================================")

if duplicate_websites.empty:
    print("No duplicate websites found.")
else:
    print(
        duplicate_websites[
            ["S/NO.", "Name", "Website", "Country"]
        ].to_string(index=False)
    )

print("\n========================================")
print("COMPLETE DUPLICATE ROW CHECK")
print("========================================")

print(
    "Completely duplicated rows:",
    final_dataset.duplicated().sum()
)

print("\n========================================")
print("N/A COUNTS")
print("========================================")

na_counts = (
    final_dataset
    .astype(str)
    .apply(lambda col: col.str.strip().str.upper().eq("N/A"))
    .sum()
)

print(na_counts)


print("\n========================================")
print("ACTUAL NaN CHECK")
print("========================================")

print(
    "Total actual NaN values:",
    final_dataset.isna().sum().sum()
)

print("\nNaN by column:")
print(final_dataset.isna().sum())

print("\n========================================")
print("COUNTRY COUNTS")
print("========================================")

print(final_dataset["Country"].value_counts())

print("\n========================================")
print("FINAL DATASET SIZE")
print("========================================")

print("Rows:", len(final_dataset))
print("Columns:", len(final_dataset.columns))

print("\n========================================")
print("S/NO. CHECK")
print("========================================")

print("Missing S/NO.:", final_dataset["S/NO."].isna().sum())
print("Duplicate S/NO.:", final_dataset["S/NO."].duplicated().sum())
print(
    "Correct sequence:",
    list(final_dataset["S/NO."]) == list(range(1, len(final_dataset) + 1))
)

print("\n========================================")
print("FINAL DATASET")
print("========================================")

display(final_dataset)

final_dataset.to_csv(
    "Nigeria_England_Wales_Health_Providers_FINAL.csv",
    index=False
)

final_dataset.to_excel(
    "Nigeria_England_Wales_Health_Providers_FINAL.xlsx",
    index=False
)

print("\n========================================")
print("FILES SAVED")
print("========================================")

print("Nigeria_England_Wales_Health_Providers_FINAL.csv")
print("Nigeria_England_Wales_Health_Providers_FINAL.xlsx")

Rows before duplicate check: 117
Rows after duplicate check: 117

DUPLICATE NAME CHECK
No duplicate organisation names found.

DUPLICATE WEBSITE CHECK
 S/NO.                                        Name                       Website Country
    14          DEFENCE HEALTH MAINTENANCE LIMITED           https://doheec.com/ Nigeria
    16 DOHEEC INTERNATIONAL HEALTHCARE LIMITED HMO           https://doheec.com/ Nigeria
    24            HALLMARK HEALTH SERVICES LIMITED https://healthpartnersng.org/ Nigeria
    26                     Health Partners Limited https://healthpartnersng.org/ Nigeria

COMPLETE DUPLICATE ROW CHECK
Completely duplicated rows: 0

N/A COUNTS
S/NO.                 0
Name                  0
HMO ID               26
Website               2
Address              27
Email                18
Contact Number       12
Country               0
Source                0
Source URL            0
State                 5
Region               32
Coverage State        0
Coverage Type       

C:\Users\user\AppData\Local\Temp\ipykernel_20208\1681747275.py:35: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  final_dataset.loc[


,S/NO.,Name,HMO ID,Website,Address,Email,Contact Number,Country,Source,Source URL,State,Region,Coverage State,Coverage Type,Services Rendered,Target Customers
0,1,A&M HEALTHCARE TRUST LIMITED,102.0,https://amhmo.com/,"Plot U Bekaji Road, Opposite Bekaji Jumma'at Mosque, Jimeta, Yola, Adamawa State",info@amhmo.com,"08033646497, 09162788582, 08024143666",Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,Adamawa,North East,Adamawa,Individual / Family / Corporate,Outpatient care; Specialist consultation; Inpatient care; Diagnostic services; Prescription medicines; Emergency care; Maternity care,Individuals; Families; Corporate Organisations
1,2,AIICO MULTISHIELD NIGERIA LIMITED,6.0,https://www.aiicomultishield.com/,"322, IKORODU ROAD, ANTHONY",info@aiicomultishield.com loshunniyi@aiicomultishield.com,07026744353 08056744353,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,Lagos,South West,Nationwide – Nigeria,Individual / Family / Corporate / SME / Group / International,Medical consultation; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Hospital care; Emergency care; Maternity care,Individuals; Families; SMEs; Corporate Organisations; Associations; Large Groups
2,3,ALLEANZA HEALTH MANAGEMENT LIMITED,111.0,https://alleanzahealth.com/individual-plans/,"? 83B, Basheer Shittu Avenue, Magodo Estate, Phase 2, Shangisha, Lagos",info@alleanzahealth.com,07036592835,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,Lagos,South West,Nationwide – Nigeria,Individual / Family / Corporate / Elderly / SME,Medical consultation; Hospital admission; Surgery; Maternity care; Emergency care; Pharmacy services; Provider network access,Individuals; Families; Elderly/Senior Citizens; SMEs; Corporate Organisations
3,4,ALLY HEALTHCARE LIMITED,119.0,https://amanhmo.com/home,"Suit C4 Plot 1196 Ndjamena Crescent, Wuse II, Abuja",allyhealthcarelimited@gmail.com,08148808171,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,FCT,North Central,FCT,Individual / Corporate,Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care,Individuals; Families; Corporate Organisations
4,5,AMAN HEALTH MAINTENANCE ORGANIZATIONS,121.0,https://anchorhmo.com/,"? 1, M. M. Alkali Street, Off 442 Crescent, Citec Vilas Gwarinpa, Abuja",info@amanhmo.com,08100588906,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,FCT,North Central,FCT,Individual / Family / Corporate,Health insurance; Managed healthcare; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care,Individuals; Families; Corporate Organisations
5,6,ANCHOR HMO INTERNATIONAL COMPANY LIMITED,76.0,https://ashmedintegratedhealthservices.com/,GOLDCREST MALL BUILDING,info@anchorhmo.com,"08031230306, 07058890062, 07080601192",Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,N/A,N/A,Nationwide – Nigeria,Individual / Corporate / SME,Corporate health plans; Retail health plans; Third-party administration; Customized health plans; Emergency support; Claims management,Individuals; Families; SMEs; Small and Large Employers; Corporate Organisations
6,7,ASHMED INTEGRATED HEALTH SERVICES LTD.,89.0,https://www.avonhealthcare.com/,"RE 013 ZUNGERU CLOSE BY BIMA ROAD NITEL QUARTERS, KADUNA",info@ashmedintegratedhealthservices.com,07036035182,Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,Kaduna,North West,Kaduna,Individual / Family / Corporate,Managed healthcare; Health insurance; Outpatient care; Inpatient care; Specialist care; Diagnostic services; Prescription medicines; Emergency care,Individuals; Families; Corporate Organisations
7,8,AVON HEALTHCARE LIMITED,63.0,https://www.axamansard.com/health/plans/,"22B, GLOVER ROAD",info@avonhealthcare.com,"08102659972, 07002779800",Nigeria,NHIA,https://www.nhia.gov.ng/hmo/,Lagos,South West,Nationwide – Nigeria,Individual / Family / Corporate,Health insurance; Outpatient care; Inpatient care; Specialist consultation; Diagnostic services; Prescription medicines; E


FILES SAVED
Nigeria_England_Wales_Health_Providers_FINAL.csv
Nigeria_England_Wales_Health_Providers_FINAL.xlsx
